# SwiftBite Capstone Project: Business Analysis & Customer Experience 

* Variant ID: SB-P19
---


### 1. Konteks Bisnis & Misi Proyek
SwiftBite adalah platform pemesanan dan pengantaran makanan yang menghubungkan pelanggan dengan restoran mitra. Tim Analytics sudah memiliki data transaksi dan pengantaran yang terintegrasi, tetapi masih perlu memahami pengalaman pelanggan melalui data Reviews.

### Tiga Area Masalah yang Perlu Dieksplorasi:
1. **Performa bisnis:** Bagaimana perubahan volume order, pembayaran bersih, dan performa transaksi selama periode data?
2. **Performa pengantaran:** Kelompok mana yang menunjukkan keterlambatan tinggi dan layak dievaluasi lebih lanjut?
3. **Customer experience:** Apa yang terlihat dari rating, isi review, tingkat partisipasi ulasan, dan sentimen pelanggan?

### Misi Resmi Capstone:
1. Merancang minimal tiga pertanyaan bisnis yang relevan dengan kondisi SwiftBite.
2. Melakukan integrasi dan validasi Reviews tanpa mengubah grain order.
3. Menganalisis metrik transaksi, pengantaran, dan pengalaman pelanggan.
4. Menghasilkan 3–5 temuan tervalidasi beserta rekomendasi bisnis.
5. Menyampaikan hasil lewat Analysis Report, Dashboard HTML, CSV, dan Technical Evidence.

### 2. Sumber Data yang Diproses (Varian SB-P19)
- `swiftbite_pm2_integrated_order_level.csv`: Hasil akhir PM2 (sumber order, pembayaran, pelanggan aman, restoran, dan pengantaran).
- `swiftbite_reviews.csv`: Handoff Tracker varian SB-P18 (sumber rating, review_timestamp, review_text, dan relasi order).

### 3. Delapan Tahap Alur Kerja Resmi Capstone
Mengacu pada panduan resmi, analisis diselesaikan melalui 8 tahap berurutan:
1. **Tahap 01 - Data Intake & Baseline:** Memeriksa struktur, jumlah baris baseline PM2, dan file Reviews SB-P18 secara aman (*read-only*).
2. **Tahap 02 - Integrasi PM2 + Reviews (Join Reviews):** Melakukan *LEFT JOIN* berbasis `order_id` tanpa penambahan baris (*no fan-out*) dan menangani order tanpa ulasan menjadi `No Review`.
3. **Tahap 03 - Business Questions & 4 KPI Inti:** Merumuskan minimal 3 pertanyaan bisnis dan menghitung formula KPI (*Total Orders*, *Net Payment Value*, *Review Coverage*, *Late Delivery Rate*).
4. **Tahap 04 - Text & Sentiment Analysis:** Mengklasifikasikan ulasan ke dalam 5 label resmi (*Positive*, *Neutral*, *Negative*, *Unknown*, *No Text*), validasi sampel manual, dan mengekspor `swiftbite_capstone_analysis_ready.csv`.
5. **Tahap 05 - Business Insights & Temuan:** Menyusun 3–5 temuan tervalidasi dengan struktur 5 elemen (*Temuan, Evidence, Interpretasi, Rekomendasi, Keterbatasan*).
6. **Tahap 06 - Dashboard HTML Interaktif:** Membangun antarmuka dashboard mandiri (minimal 4 KPI, 3 chart, 2 filter fungsional) yang dapat dibuka langsung tanpa server Python.
7. **Tahap 07 - Final QA, Packaging & Technical Evidence:** Menjalankan verifikasi akhir konsistensi data antar-artefak dan mengemas paket pengumpulan ZIP.
8. **Tahap 08 - Showcase & Defense Preparation:** Menyiapkan alur narasi presentasi laporan analitik untuk tahap evaluasi akhir.


## Tahap 0 Setup Identitas & Folder Workspace
Tujuan: Menyiapkan kebutuhan awal analisis data SwiftBite, mulai dari mengatur library, menentukan lokasi sumber data, membuat folder penyimpanan, mencatat proses kerja, hingga memastikan file yang dibutuhkan tersedia sebelum analisis dilakukan.

In [2]:
!pip install pandas numpy matplotlib

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.3 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.3 MB 2.0 MB/s eta 0:00:05
   ---- ----------------------------------- 1.0/9.3 MB 2.1 MB/s eta 0:00:05
   ------ --------------------------------- 1.6/9.3 MB 2.0 MB/s eta 0:00:04
   -------- ------------------------------- 2.1/9.3 MB 2.1 MB/s eta 0:00:04
   ----------- ---------------------------- 2.6/9.3 MB 2.2 MB/s eta 0:00:04
   ------------- -------------------------- 3.1/9.3 MB 2.3 MB/s eta 0:00:03
   ---------------- ----------------------- 3.9/9.3 MB 2.3 MB/s eta 0:00:03
   ----------------- ---------------------- 4.2/9.3 MB 2.3 MB/s eta 0:00:03
   --------------------- ------------------ 5.0/9.3 MB 2.4 MB/s eta 0:00:02
   ----------------------- ---------------- 5.5/9.3 MB 2.4 MB/s eta 0:00:02
   ---------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\hp\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import json, re, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

# ====== EDIT BAGIAN INI ======
VARIANT_ID  = "SB-P19"          # ganti sesuai Handoff Tracker
PARTICIPANT = "AzizahGhina"     # nama kamu (tanpa spasi)
BASE = Path("..").resolve()     # = Bootcamp_Data_Analyst/

PM2_PATH     = BASE / "data/processed/pm2_swiftbite/swiftbite_pm2_integrated_order_level.csv"
REVIEWS_PATH = BASE / "data/raw/capstone_swiftbite/swiftbite_reviews.csv"
# =============================

OUT_PROCESSED = BASE / "data/processed/capstone_swiftbite"
OUT_DASH      = BASE / "outputs/capstone_swiftbite"
OUT_DOCS      = BASE / "docs/capstone_swiftbite"
OUT_FIG       = OUT_DOCS / "figures"
for p in (OUT_PROCESSED, OUT_DASH, OUT_DOCS, OUT_FIG):
    p.mkdir(parents=True, exist_ok=True)

EVIDENCE = {"variant_id": VARIANT_ID}   # semua angka validasi dikumpulkan di sini
LOG = []                                 # catatan perubahan/keputusan (untuk Technical Evidence)

def log(msg):
    LOG.append(msg)
    print("[LOG]", msg)

print("PM2     :", PM2_PATH, "Ditemukan" if PM2_PATH.exists() else "Tidak Ditemukan")
print("Reviews :", REVIEWS_PATH, "Ditemukan" if REVIEWS_PATH.exists() else "Tidak Ditemukan")

PM2     : C:\Users\hp\OneDrive\Dokumen\Future Maker Programe\Bootcamp_Data_Analyst\data\processed\pm2_swiftbite\swiftbite_pm2_integrated_order_level.csv Ditemukan
Reviews : C:\Users\hp\OneDrive\Dokumen\Future Maker Programe\Bootcamp_Data_Analyst\data\raw\capstone_swiftbite\swiftbite_reviews.csv Ditemukan


## Tahap 1 — Data Intake 

Tujuan: memastikan dua input terbaca, skemanya dipahami, dan baseline PM2 tercatat.
Baseline jumlah baris dan `order_id` unik PM2 **tidak boleh berubah** setelah join.

Input:
1. Dataset akhir PM2 (swiftbite_pm2_integrated_order_level.csv)  
2. Dataset ulasan (swiftbite_reviews.csv).   
3. Varian ID SB-P19

Yang dilakukan : 
1. Membaca dan memeriksa struktur dataset.
2. Membersihkan spasi pada kolom order_id.
3. Memeriksa jumlah baris, ID unik, nilai kosong, dan duplikasi.
4. Mencocokkan order_id antara PM2 dan Reviews.
5. Memeriksa kolom yang berpotensi mengandung PII.
6. Memvalidasi identitas varian dan menyimpan hasil pemeriksaan.
 

In [2]:
pm2     = pd.read_csv(PM2_PATH, dtype=str, keep_default_na=False, na_values=[""])
reviews = pd.read_csv(REVIEWS_PATH, dtype=str, keep_default_na=False, na_values=[""])

for name, df in (("PM2", pm2), ("REVIEWS", reviews)):
    print(f"=== {name}: {df.shape[0]:,} baris x {df.shape[1]} kolom ===")
    print(list(df.columns))
    print()

display(pm2.head(3))
display(reviews.head(3))

=== PM2: 335 baris x 40 kolom ===
['order_id', 'restaurant_id', 'order_timestamp', 'order_status', 'promo_code', 'subtotal_amount', 'delivery_fee', 'discount_amount', 'total_amount', 'customer_key', 'signup_date', 'customer_zone', 'customer_segment', 'restaurant_name', 'cuisine_type', 'restaurant_zone', 'price_tier', 'onboarding_date', 'active_status', 'item_count', 'total_quantity', 'distinct_item_category_count', 'items_subtotal', 'payment_attempt_count', 'successful_payment_amount', 'refunded_payment_amount', 'net_payment_amount', 'final_payment_status', 'payment_method_final', 'delivery_id', 'courier_id', 'assigned_at', 'picked_up_at', 'delivered_at', 'delivery_status', 'distance_km', 'promised_delivery_minutes', 'actual_delivery_minutes', 'delay_minutes', 'is_late']

=== REVIEWS: 192 baris x 5 kolom ===
['review_id', 'order_id', 'review_timestamp', 'rating', 'review_text']



,order_id,restaurant_id,order_timestamp,order_status,promo_code,subtotal_amount,delivery_fee,discount_amount,total_amount,customer_key,signup_date,customer_zone,customer_segment,restaurant_name,cuisine_type,restaurant_zone,price_tier,onboarding_date,active_status,item_count,total_quantity,distinct_item_category_count,items_subtotal,payment_attempt_count,successful_payment_amount,refunded_payment_amount,net_payment_amount,final_payment_status,payment_method_final,delivery_id,courier_id,assigned_at,picked_up_at,delivered_at,delivery_status,distance_km,promised_delivery_minutes,actual_delivery_minutes,delay_minutes,is_late
0,O00001,R0019,2026-03-28 11:18:00,COMPLETED,SAVE10,136000,8000,10000,134000,38c9e42f95f9d528e9afa9f145c54bfc6814b19420f6ca337a92d8966d69e8c6,2025-12-06,South,New,SwiftBite Partner 19,Western,North,BUDGET,2024-09-10,inactive,3,5,1,136000,1,134000,0,134000,successful,E_Wallet,D00001,CR003,2026-03-28 11:26:00,2026-03-28 11:40:00,2026-03-28 12:10:00,delivered,0.9,44.0,44.0,0.0,False
1,O00002,R0001,2026-02-09 04:58:00,COMPLETED,NaN,106000,12000,10000,108000,68d877c4e3c016d8bbf29e8c80e9365115342f6bdb8d22e5ee8fdab766995fb9,2025-08-06,West,Regular,SwiftBite Partner 01,Western,East,MID,2025-01-03,active,2,6,2,106000,1,108000,0,108000,successful,Bank_Transfer,D00002,CR013,2026-02-09 05:05:00,2026-02-09 05:19:00,2026-02-09 05:56:00,delivered,7.8,55.0,51.0,-4.0,False
2,O00003,R0006,2026-03-27 21:11:00,COMPLETED,NaN,66000,18000,15000,69000,4ed9c844e56b7399f1f76ad50f9a8c900c4a394da5cf17c43b96627a3d31e7ec,2025-07-08,West,Loyal,SwiftBite Partner 06,Asian,Central,BUDGET,2025-01-12,active,1,3,1,66000,1,69000,0,69000,successful,Credit_Card,D00003,CR013,2026-03-27 21:18:00,2026-03-27 21:41:00,2026-03-27 22:26:00,delivered,3.8,53.0,68.0,15.0,True


,review_id,order_id,review_timestamp,rating,review_text
0,RV00001,O00248,2026-03-24 23:39:00,2,Kurir sulit dihubungi saat pengantaran.
1,RV00002,O00153,2026-02-06 05:33:00,4,"Driver ramah, tetapi makanannya sudah tidak terlalu hangat."
2,RV00003,O00272,2026-03-16 08:15:00,4,Pengiriman biasa saja dan sesuai perkiraan.


In [3]:
# --- Pemeriksaan format key (periksa dulu, ubah hanya jika terbukti perlu) ---
assert "order_id" in pm2.columns,     "PM2 tidak punya kolom order_id"
assert "order_id" in reviews.columns, "Reviews tidak punya kolom order_id"

pm2["order_id"]     = pm2["order_id"].astype("string")
reviews["order_id"] = reviews["order_id"].astype("string")

def count_whitespace_keys(s):
    """Jumlah key yang punya spasi di awal/akhir."""
    return int((s != s.str.strip()).fillna(False).sum())

ws_pm2 = count_whitespace_keys(pm2["order_id"])
ws_rev = count_whitespace_keys(reviews["order_id"])
print(f"Key dengan spasi tersembunyi -> PM2: {ws_pm2} | Reviews: {ws_rev}")
EVIDENCE.update(pm2_key_whitespace=ws_pm2, reviews_key_whitespace=ws_rev)

if ws_pm2 or ws_rev:
    # Temuan validasi: dicatat, lalu diperbaiki hanya pada key (nilai bisnis tidak disentuh)
    pm2["order_id"]     = pm2["order_id"].str.strip()
    reviews["order_id"] = reviews["order_id"].str.strip()
    log(f"Ditemukan spasi tersembunyi pada order_id (PM2={ws_pm2}, Reviews={ws_rev}); "
        f"di-trim di memori saja. File sumber tidak diubah.")
else:
    log("Pemeriksaan format key: tidak ada spasi tersembunyi di PM2 maupun Reviews; tidak ada transformasi pada key.")

# --- Baseline PM2 ---
pm2_rows   = len(pm2)
pm2_unique = pm2["order_id"].nunique(dropna=True)
pm2_null   = int(pm2["order_id"].isna().sum())
pm2_dup    = int(pm2["order_id"].duplicated(keep=False).sum())

# --- Profil Reviews ---
rev_rows   = len(reviews)
rev_unique = reviews["order_id"].nunique(dropna=True)
rev_null   = int(reviews["order_id"].isna().sum())
rev_dup    = int(reviews["order_id"].duplicated(keep=False).sum())

print(f"PM2     : rows={pm2_rows:,} | unique order_id={pm2_unique:,} | null={pm2_null} | baris duplikat={pm2_dup}")
print(f"Reviews : rows={rev_rows:,} | unique order_id={rev_unique:,} | null={rev_null} | baris duplikat={rev_dup}")

# --- Kecocokan key ---
pm2_ids = set(pm2["order_id"].dropna())
rev_ids = set(reviews["order_id"].dropna())
matched_ids   = rev_ids & pm2_ids
unmatched_ids = rev_ids - pm2_ids
print(f"Review order_id yang cocok dengan PM2 : {len(matched_ids):,}")
print(f"Review order_id TIDAK ada di PM2      : {len(unmatched_ids):,}")
if unmatched_ids:
    print("   contoh:", list(unmatched_ids)[:5])
    print("   -> cek lagi apakah Reviews yang dipakai sesuai Variant ID, dan format key (spasi/huruf besar/kecil/leading zero).")

# --- Pemeriksaan Safe Data pada PM2 (input) ---
PII_REGEX = re.compile(r"full_name|e[-_]?mail|phone|salt|^customer_id$", re.I)
pii_cols_pm2 = [c for c in pm2.columns if PII_REGEX.search(c)]
print("Kolom PII/terlarang di PM2:", pii_cols_pm2 or "tidak ada")

# --- Cek identitas varian (jika ada kolomnya) ---
for name, df in (("PM2", pm2), ("Reviews", reviews)):
    vcols = [c for c in df.columns if "variant" in c.lower()]
    for c in vcols:
        print(f"{name}.{c} unik:", df[c].dropna().unique()[:5])
print(f"Pastikan Variant ID = {VARIANT_ID} sama dengan Handoff Tracker dan nama file Reviews.")

EVIDENCE.update(pm2_rows=pm2_rows, pm2_unique_order_id=pm2_unique, pm2_null_order_id=pm2_null,
                pm2_dup_order_id=pm2_dup, reviews_rows=rev_rows, reviews_unique_order_id=rev_unique,
                reviews_dup_order_id=rev_dup, matched_review_order_id=len(matched_ids),
                unmatched_reviews=len(unmatched_ids), pm2_pii_cols=pii_cols_pm2)

assert pm2_null == 0 and pm2_dup == 0, "STOP: order_id PM2 null/duplikat. Selesaikan di PM2, bukan di Capstone."
assert pm2_rows == pm2_unique, "STOP: PM2 bukan satu baris per order."
assert not pii_cols_pm2, f"STOP: PM2 memuat kolom terlarang: {pii_cols_pm2}"

Key dengan spasi tersembunyi -> PM2: 0 | Reviews: 0
[LOG] Pemeriksaan format key: tidak ada spasi tersembunyi di PM2 maupun Reviews; tidak ada transformasi pada key.
PM2     : rows=335 | unique order_id=335 | null=0 | baris duplikat=0
Reviews : rows=192 | unique order_id=192 | null=0 | baris duplikat=0
Review order_id yang cocok dengan PM2 : 192
Review order_id TIDAK ada di PM2      : 0
Kolom PII/terlarang di PM2: tidak ada
Pastikan Variant ID = SB-P19 sama dengan Handoff Tracker dan nama file Reviews.


## Tahap 2 - Join Reviews (Integrasi Data)
Tujuan: Menggabungkan data ulasan pelanggan (Reviews) ke dalam dataset final PM2 tanpa merusak struktur data (grain).   

Ketentuan Teknis:
1. Menggunakan metode LEFT JOIN dari tabel utama PM2 ke Reviews dengan key penghubung order_id.
2. Memastikan tidak terjadi fan-out (jumlah baris akhir harus tetap sama persis dengan jumlah baris PM2, yaitu satu baris per order).   
3. Order yang tidak memiliki ulasan tetap dipertahankan dengan status khusus (No Review).  

Yang dilakukan: 
1. Membersihkan dan menstandarisasi kolom rating serta teks ulasan.
2. Melakukan audit 6 pemeriksaan wajib (jumlah baris, keunikan ID, nominal uang, dll.) lewat tabel validasi.
3. Menyimpan hasilnya ke file kerja sementara (swiftbite_capstone_working.csv) dan mencatat log status akhirnya (PASS / NEEDS FIX).

In [4]:
def pick(df, candidates, required=True, label=""):
    """Cari kolom: cocok persis dulu, lalu cocok sebagian (case-insensitive)."""
    cols = list(df.columns)
    low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    for cand in candidates:
        for c in cols:
            if cand.lower() in c.lower():
                return c
    if required:
        raise KeyError(f"Kolom '{label or candidates[0]}' tidak ditemukan. Kolom tersedia: {cols}")
    return None

R_RATING = pick(reviews, ["rating", "review_rating", "score"], label="rating")
R_TEXT   = pick(reviews, ["review_text", "review", "comment", "text"], label="review_text")
R_TS     = pick(reviews, ["review_timestamp", "review_date", "created_at"], required=False)
print("Kolom Reviews terdeteksi -> rating:", R_RATING, "| text:", R_TEXT, "| timestamp:", R_TS)

rev = reviews[["order_id"] + [c for c in (R_RATING, R_TEXT, R_TS) if c]].copy()
rev = rev.rename(columns={R_RATING: "rating", R_TEXT: "review_text", **({R_TS: "review_timestamp"} if R_TS else {})})
rev["rating"] = pd.to_numeric(rev["rating"], errors="coerce")

# bersihkan teks "kosong" tanpa mengubah teks yang ada isinya
def clean_text(x):
    if pd.isna(x):
        return pd.NA
    t = str(x).strip()
    return pd.NA if t.lower() in {"", "nan", "none", "null", "n/a", "-"} else t
rev["review_text"] = rev["review_text"].map(clean_text).astype("string")

# Relasi harus 1 review per order (sesuai paket resmi). Jika tidak, STOP dan telusuri.
dups = rev[rev["order_id"].duplicated(keep=False)].sort_values("order_id")
if len(dups):
    display(dups.head(20))
    raise AssertionError(f"STOP: {dups['order_id'].nunique()} order_id punya >1 review. Telusuri dulu, jangan langsung drop.")

rev["_in_reviews"] = True
merged = pm2.merge(rev, on="order_id", how="left", validate="one_to_one")
merged["has_review"] = merged["_in_reviews"].fillna(False).astype(bool).astype(int)
merged = merged.drop(columns="_in_reviews")

def _status(r):
    if r["has_review"] == 0:
        return "No Review"
    return "Reviewed - No Text" if pd.isna(r["review_text"]) else "Reviewed - With Text"
merged["review_status"] = merged.apply(_status, axis=1)

print(merged["review_status"].value_counts())
print("Baris setelah join:", len(merged), "| PM2:", pm2_rows)
assert len(merged) == pm2_rows, "STOP: jumlah baris berubah setelah join (fan-out)!"

Kolom Reviews terdeteksi -> rating: rating | text: review_text | timestamp: review_timestamp
review_status
Reviewed - With Text    181
No Review               143
Reviewed - No Text       11
Name: count, dtype: int64
Baris setelah join: 335 | PM2: 335


In [5]:
# --- Enam pemeriksaan wajib: tulis angka aktual ---
# Kolom pembayaran (untuk check #5) - dideteksi otomatis, cek hasilnya.
pay_cols = [c for c in pm2.columns if re.search(r"payment_amount|paid_amount|amount", c, re.I)]
print("Kolom nominal yang diperiksa:", pay_cols)

def total_money(df):
    return {c: pd.to_numeric(df[c], errors="coerce").sum() for c in pay_cols}

before_money = total_money(pm2)
after_money  = total_money(merged)

pii_after = [c for c in merged.columns if PII_REGEX.search(c)]
checks = [
    ("1. Row count",        f"PM2={pm2_rows} | Capstone={len(merged)}", len(merged) == pm2_rows),
    ("2. Unique order_id",  f"unique={merged['order_id'].nunique()} | null={merged['order_id'].isna().sum()} | dup={merged['order_id'].duplicated().sum()}",
        merged["order_id"].nunique() == len(merged) and merged["order_id"].isna().sum() == 0),
    ("3. Relationship",     f"matched={len(matched_ids)} | unmatched reviews={len(unmatched_ids)}", len(unmatched_ids) == 0),
    ("4. Missing review",   f"No Review={int((merged['review_status']=='No Review').sum())} (tetap dipertahankan)", True),
    ("5. Monetary values",  "sebelum=" + str({k: round(v, 2) for k, v in before_money.items()}) + " | sesudah=" + str({k: round(v, 2) for k, v in after_money.items()}),
        all(np.isclose(before_money[k], after_money[k]) for k in pay_cols)),
    ("6. Safe Data",        f"direct PII field count={len(pii_after)} {pii_after}", len(pii_after) == 0),
]
chk = pd.DataFrame(checks, columns=["check", "actual_result", "pass"])
chk["status"] = np.where(chk["pass"], "PASS", "NEEDS FIX")
display(chk[["check", "actual_result", "status"]])

EVIDENCE.update(capstone_rows=len(merged), unique_order_id=int(merged["order_id"].nunique()),
                duplicate_order_id=int(merged["order_id"].duplicated().sum()),
                direct_pii_field_count=len(pii_after),
                no_review_orders=int((merged["review_status"] == "No Review").sum()),
                reviewed_no_text=int((merged["review_status"] == "Reviewed - No Text").sum()),
                reviewed_with_text=int((merged["review_status"] == "Reviewed - With Text").sum()),
                money_before=before_money, money_after=after_money,
                join_judgment="PASS" if chk["pass"].all() else "NEEDS FIX")
print("\nFinal judgment:", EVIDENCE["join_judgment"])
assert chk["pass"].all(), "Ada check yang gagal. Perbaiki dulu (lihat tabel), simpan evidence sebelum & sesudah perbaikan."

Kolom nominal yang diperiksa: ['subtotal_amount', 'discount_amount', 'total_amount', 'successful_payment_amount', 'refunded_payment_amount', 'net_payment_amount']


,check,actual_result,status
0,1. Row count,PM2=335 | Capstone=335,PASS
1,2. Unique order_id,unique=335 | null=0 | dup=0,PASS
2,3. Relationship,matched=192 | unmatched reviews=0,PASS
3,4. Missing review,No Review=143 (tetap dipertahankan),PASS
4,5. Monetary values,"sebelum={'subtotal_amount': np.int64(40199000), 'discount_amount': np.int64(...",PASS
5,6. Safe Data,direct PII field count=0 [],PASS



Final judgment: PASS


In [6]:
# Simpan working dataset (BUKAN final submission). Sumber tidak ditimpa.
working_path = OUT_PROCESSED / "swiftbite_capstone_working.csv"
merged.to_csv(working_path, index=False)
log(f"LEFT JOIN PM2->Reviews on order_id (validate one_to_one). rows {pm2_rows}->{len(merged)}. Disimpan: {working_path}")

[LOG] LEFT JOIN PM2->Reviews on order_id (validate one_to_one). rows 335->335. Disimpan: C:\Users\hp\OneDrive\Dokumen\Future Maker Programe\Bootcamp_Data_Analyst\data\processed\capstone_swiftbite\swiftbite_capstone_working.csv


## Tahap 3 Business Analytics (Analisis Bisnis & KPI)

Tujuan: Menjawab minimal 3 pertanyaan bisnis yang mencakup area performa bisnis, operasional pengantaran, dan pengalaman pelanggan.   

| # | Area | Business Question | Data | KPI / Metode | Visual | Keputusan bisnis |
|---|---|---|---|---|---|---|
| BQ1 | Business performance | Bagaimana perubahan Total Orders, Net Payment Value, dan pembatalan per bulan (Jan-Mar 2026)? | order_timestamp, order_status, net_payment_amount | COUNT DISTINCT order_id; SUM sukses - SUM refund; cancel rate per bulan| Bar/line per bulan | Kapan volume naik/turun, bulan mana perlu ditelaah |
| BQ2 | Delivery performance | Restoran/kelompok mana yang Late Delivery Rate-nya tinggi dengan jumlah delivered valid yang memadai? | is_late, delivery_status, customer_zone, restaurant_zone, price_tier | Late / delivered valid, tampilkan N | Bar per restoran + tabel N | Prioritas evaluasi estimasi waktu & alokasi |
| BQ3 | Customer experience | Apa tema keluhan dan pola rating vs sentimen, serta partisipasi review? | rating, review_text, review_status, sentiment_label | Review Coverage; distribusi sentimen; crosstab rating x sentimen; frekuensi tema | Bar sentimen + crosstab rating vs sentimen | Memfokuskan perbaikan pada tema keluhan terbanyak |
| BQ3 | Penghubung | Apakah order terlambat cenderung mendapat rating lebih rendah? | is_late, rating | Rata-rata rating terlambat vs tepat waktu dengan n | Bar perbandingan | Menjadi dasar investigasi lanjutan keterlambatan terhadap kepuasan (bukan sebab-akibat) |

In [7]:
# --- Deteksi kolom analisis di PM2. Cek hasilnya; jika salah, isi OVERRIDE. ---
CANDIDATES = {
    "order_ts":        ["order_date", "order_timestamp", "order_created_at", "order_datetime", "created_at", "order_time"],
    "restaurant":      ["restaurant_name", "restaurant_id", "restaurant_key", "restaurant"],
    "success_amt":     ["successful_payment_amount", "success_payment_amount", "success_amount", "paid_amount"],
    "refund_amt":      ["refunded_payment_amount", "refund_payment_amount", "refund_amount", "refunded_amount"],
    "delivery_status": ["delivery_status", "order_status", "status"],
    "is_late":         ["is_late", "late_flag", "late_delivery", "is_delayed", "delay_flag"],
    "est_ts":          ["estimated_delivery", "estimated_delivery_time", "promised_delivery", "expected_delivery", "eta"],
    "act_ts":          ["actual_delivery", "actual_delivery_time", "delivered_at", "delivery_time"],
}
OVERRIDE = {}   # contoh: {"restaurant": "restaurant_name", "order_ts": "order_date"}

COL = {}
for key, cands in CANDIDATES.items():
    COL[key] = OVERRIDE.get(key) or pick(merged, cands, required=False)
print(json.dumps(COL, indent=2))
for must in ("order_ts", "success_amt"):
    assert COL[must], f"Kolom '{must}' belum terdeteksi. Isi OVERRIDE['{must}']. Kolom tersedia: {list(merged.columns)}"
assert COL["restaurant"], f"Kolom restoran belum terdeteksi. Isi OVERRIDE['restaurant']. Kolom tersedia: {list(merged.columns)}"
assert COL["is_late"] or (COL["est_ts"] and COL["act_ts"]), "Perlu kolom is_late ATAU pasangan est_ts + act_ts untuk Late Delivery Rate."

{
  "order_ts": "order_timestamp",
  "restaurant": "restaurant_name",
  "success_amt": "successful_payment_amount",
  "refund_amt": "refunded_payment_amount",
  "delivery_status": "delivery_status",
  "is_late": "is_late",
  "est_ts": "promised_delivery_minutes",
  "act_ts": "delivered_at"
}


In [9]:
df = merged.copy()

# --- Turunan tanggal ---
df["order_ts_parsed"] = pd.to_datetime(df[COL["order_ts"]], errors="coerce")
df["month"] = df["order_ts_parsed"].dt.strftime("%Y-%m").fillna("Unknown")
print("Tanggal tidak terbaca:", int(df["order_ts_parsed"].isna().sum()), "| rentang:", df["order_ts_parsed"].min(), "->", df["order_ts_parsed"].max())

# --- Pembayaran ---
df["success_amt"] = pd.to_numeric(df[COL["success_amt"]], errors="coerce").fillna(0.0)
df["refund_amt"]  = pd.to_numeric(df[COL["refund_amt"]], errors="coerce").fillna(0.0) if COL["refund_amt"] else 0.0
df["net_payment"] = df["success_amt"] - df["refund_amt"]       # failed attempts tidak ikut karena hanya memakai kolom success

# --- Pengantaran ---
BOOL_MAP = {"1": 1, "true": 1, "yes": 1, "y": 1, "late": 1, "0": 0, "false": 0, "no": 0, "n": 0, "on time": 0, "on_time": 0, "ontime": 0}
status = df[COL["delivery_status"]].astype("string").str.lower().str.strip() if COL["delivery_status"] else None
act = pd.to_datetime(df[COL["act_ts"]], errors="coerce") if COL["act_ts"] else None
est = None if COL["is_late"] else (pd.to_datetime(df[COL["est_ts"]], errors="coerce") if COL["est_ts"] else None)

if status is not None:
    df["delivered"] = status.str.contains("deliver", na=False) & ~status.str.contains("undeliver|not deliver", na=False)
elif act is not None:
    df["delivered"] = act.notna()
else:
    raise ValueError("Tidak ada kolom status/waktu aktual untuk menentukan delivered.")

if COL["is_late"]:
    late = df[COL["is_late"]].astype("string").str.lower().str.strip().map(BOOL_MAP)
else:
    late = pd.Series(np.where(act.notna() & est.notna(), (act > est).astype(float), np.nan), index=df.index)

df["late"] = pd.to_numeric(late, errors="coerce")
df["delivered_valid"] = (df["delivered"] & df["late"].notna()).astype(int)
df["late_flag"] = np.where(df["delivered_valid"] == 1, df["late"], np.nan)   # cancelled/incomplete TIDAK dihitung tepat waktu
df["restaurant"] = df[COL["restaurant"]].astype("string").fillna("Unknown")
print(df[["delivered", "delivered_valid"]].sum())
print("Nilai status pengantaran:", status.value_counts(dropna=False).to_dict() if status is not None else "n/a")

Tanggal tidak terbaca: 0 | rentang: 2026-01-02 22:16:00 -> 2026-03-31 04:43:00
delivered          295
delivered_valid    295
dtype: Int64
Nilai status pengantaran: {'delivered': 295, 'cancelled': 20, <NA>: 20}


In [10]:
def pct(n, d):
    return None if d == 0 else 100.0 * n / d

def kpis(d):
    """Empat KPI inti. Rasio dengan denominator 0 -> None (ditampilkan N/A)."""
    return {
        "total_orders":      int(d["order_id"].nunique()),
        "net_payment":       float(d["net_payment"].sum()),
        "review_coverage":   pct(int((d["has_review"] == 1).sum()), int(d["order_id"].nunique())),   # No Review masuk denominator
        "late_delivery_rate": pct(int(d["late_flag"].sum()) if d["delivered_valid"].sum() else 0, int(d["delivered_valid"].sum())),
        "_late_n": int(d["late_flag"].sum()) if d["delivered_valid"].sum() else 0,
        "_delivered_valid": int(d["delivered_valid"].sum()),
    }

KPI_ALL = kpis(df)
print(json.dumps(KPI_ALL, indent=2))

# KPI tambahan (opsional) - denominator dijelaskan eksplisit
rated = df["rating"].dropna()
avg_rating = float(rated.mean()) if len(rated) else None
paid_orders = int((df["success_amt"] > 0).sum())
aov = float(df.loc[df["success_amt"] > 0, "net_payment"].sum() / paid_orders) if paid_orders else None
print(f"Average Rating (review dg rating valid, n={len(rated)}): {avg_rating}")
print(f"AOV (net payment / order dg success_amt > 0, n={paid_orders}): {aov}")

EVIDENCE["kpi_unfiltered"] = KPI_ALL
EVIDENCE["kpi_definitions"] = {
    "Total Orders": "COUNT DISTINCT order_id pada populasi terpilih",
    "Net Payment Value": "SUM(successful_payment_amount) - SUM(refunded_payment_amount); failed attempts tidak menambah success",
    "Review Coverage": "(order dengan review / seluruh order) x 100%; No Review masuk denominator",
    "Late Delivery Rate": "(delivered order terlambat / delivered order dengan informasi waktu valid) x 100%; cancelled/incomplete bukan 'tepat waktu'",
    "AOV (opsional)": "net payment / order dengan success_amt > 0",
}

{
  "total_orders": 335,
  "net_payment": 37161000.0,
  "review_coverage": 57.3134328358209,
  "late_delivery_rate": 31.1864406779661,
  "_late_n": 92,
  "_delivered_valid": 295
}
Average Rating (review dg rating valid, n=192): 3.5833333333333335
AOV (net payment / order dg success_amt > 0, n=308): 120652.5974025974


In [11]:
completed = df[df["order_status"] == "COMPLETED"]
cov_completed = pct(int((completed["has_review"] == 1).sum()), len(completed))
EVIDENCE["review_coverage_completed_orders"] = cov_completed
print(f"Review Coverage KPI resmi (semua order): {KPI_ALL['review_coverage']:.1f}%")
print(f"Pembanding, order COMPLETED saja ({len(completed)} order): {cov_completed:.1f}%")

Review Coverage KPI resmi (semua order): 57.3%
Pembanding, order COMPLETED saja (295 order): 65.1%


In [13]:
# Analisis metode pembayaran (jangan ubah file PM2)
df["payment_method"] = df["payment_method_final"].replace({"Bank Transfer": "Bank_Transfer", "E Wallet": "E_Wallet"})
log("Label payment_method_final diseragamkan di memori (Bank Transfer->Bank_Transfer, E Wallet->E_Wallet), 2 baris terdampak. File sumber tidak diubah.")

[LOG] Label payment_method_final diseragamkan di memori (Bank Transfer->Bank_Transfer, E Wallet->E_Wallet), 2 baris terdampak. File sumber tidak diubah.


## Tahap 4 - Text & Sentiment Analysis (Analisis Sentimen Teks Ulasan)

Tujuan: Mengklasifikasikan isi ulasan teks pelanggan secara objektif (tidak berasumsi bahwa rating bintang 5 pasti selalu positif).   

Kategori Sentimen Resmi: Positive, Neutral, Negative, Unknown (jika teks ada tapi maknanya ambigu), dan No Text (jika baris ulasan ada tapi teksnya kosong).  

Catatan: Status No Review dipisahkan dari kategori sentimen. Dataset akhir yang sudah diperkaya dengan sentimen ini disimpan sebagai 

In [14]:
POS_WORDS = {"enak","lezat","mantap","puas","suka","bagus","cepat","ramah","hangat","segar","fresh","recommended","rekomen",
             "rekomendasi","worth","juara","sempurna","top","memuaskan","nikmat","gurih","empuk","rapi","terbaik","mantul",
             "love","great","good","delicious","tasty","fast","excellent","amazing","perfect","friendly","tepat","sesuai","lengkap",
             "terima","kasih","thanks","wangi","renyah","crispy","gercep","puas"}
NEG_WORDS = {"lama","telat","terlambat","lambat","dingin","basi","kecewa","buruk","jelek","mengecewakan","hambar","asin","rusak",
             "tumpah","bocor","mahal","parah","zonk","lelet","cold","late","slow","bad","terrible","awful","worst","disappointed",
             "stale","missing","wrong","rude","amis","gosong","keras","berantakan","hilang","batal","dibatalkan","salah","kurang",
             "complaint","komplain","refund","nyesel","menyesal","lemah","pahit","encer"}
NEG_PHRASES = ["tidak sesuai","tak sesuai","tidak lengkap","pesanan salah","salah pesan","kemasan rusak","packing rusak",
               "tidak datang","belum datang","never again","too late","not fresh","tidak ramah","kurang enak","kurang fresh"]
NEGATORS = {"tidak","tak","ga","gak","nggak","ngga","bukan","belum","nggk","not","no","never","kurang"}
NEUTRAL_MARKERS = {"biasa","standar","lumayan","cukup","average","okay","so","sedang","datar"}

TOKEN_RE = re.compile(r"[a-zA-Z']+")

def classify(text):
    """Return (label, pos_hits, neg_hits, mixed)."""
    if text is None or pd.isna(text):
        return ("No Text", 0, 0, 0)
    t = str(text).lower()
    toks = TOKEN_RE.findall(t)
    if len(toks) == 0:
        return ("Unknown", 0, 0, 0)
    pos = neg = 0
    for i, tok in enumerate(toks):
        window = toks[max(0, i - 2):i]
        flipped = any(w in NEGATORS for w in window)
        if tok in POS_WORDS:
            pos, neg = (pos, neg + 1) if flipped else (pos + 1, neg)
        elif tok in NEG_WORDS and tok != "kurang":
            pos, neg = (pos + 1, neg) if flipped else (pos, neg + 1)
    phrase_hits = sum(t.count(p) for p in NEG_PHRASES)
    neg += phrase_hits
    mixed = int(pos > 0 and neg > 0)
    if pos == 0 and neg == 0:
        if any(tok in NEUTRAL_MARKERS for tok in toks):
            return ("Neutral", 0, 0, 0)
        return ("Unknown", 0, 0, 0)
    if neg >= pos and neg > 0:
        return ("Negative", pos, neg, mixed)
    return ("Positive", pos, neg, mixed)

res = df["review_text"].apply(lambda x: classify(x if df is not None else x))
df["_sent_tuple"] = res
df["sentiment_label"] = res.map(lambda r: r[0])
df["sentiment_pos_hits"] = res.map(lambda r: r[1])
df["sentiment_neg_hits"] = res.map(lambda r: r[2])
df["sentiment_mixed"] = res.map(lambda r: r[3])
df["sentiment_method"] = "rule_based_v1"
df = df.drop(columns="_sent_tuple")

# Order tanpa record review: BUKAN sentimen -> kosongkan
df.loc[df["has_review"] == 0, ["sentiment_label", "sentiment_method"]] = np.nan
df.loc[df["has_review"] == 0, ["sentiment_pos_hits", "sentiment_neg_hits", "sentiment_mixed"]] = np.nan
# Record review ada tetapi teks kosong -> No Text
df.loc[(df["has_review"] == 1) & (df["review_text"].isna()), "sentiment_label"] = "No Text"

print(df["sentiment_label"].value_counts(dropna=False))
print("\nreview_status:\n", df["review_status"].value_counts())

sentiment_label
NaN         143
Positive     71
Negative     51
Neutral      32
Unknown      27
No Text      11
Name: count, dtype: int64

review_status:
 review_status
Reviewed - With Text    181
No Review               143
Reviewed - No Text       11
Name: count, dtype: int64


In [15]:
# Kata paling sering pada review (bantu memperkaya kamus). Tinjau, lalu tambahkan ke POS_WORDS / NEG_WORDS di sel sebelumnya.
from collections import Counter
cnt = Counter(tok for t in df["review_text"].dropna() for tok in TOKEN_RE.findall(str(t).lower()))
known = POS_WORDS | NEG_WORDS | NEGATORS | NEUTRAL_MARKERS
top_unknown = [(w, c) for w, c in cnt.most_common(200) if w not in known and len(w) > 2][:40]
print("Kata sering muncul yang BELUM ada di kamus:", top_unknown)

Kata sering muncul yang BELUM ada di kamus: [('dan', 123), ('makanannya', 42), ('terlalu', 33), ('kualitasnya', 32), ('harga', 28), ('masih', 23), ('yang', 23), ('porsinya', 22), ('pengiriman', 19), ('tapi', 17), ('terasa', 16), ('perkiraan', 16), ('saat', 15), ('delivery', 14), ('seimbang', 13), ('banyak', 13), ('oke', 13), ('spesial', 13), ('packaging', 12), ('sampai', 12), ('dari', 12), ('saya', 12), ('harapkan', 12), ('driver', 12), ('saja', 12), ('pesanan', 11), ('ada', 10), ('pengantaran', 10), ('rasanya', 10), ('konsisten', 10), ('harganya', 9), ('ekspektasi', 8), ('sudah', 8), ('kemasannya', 8), ('pengalaman', 7), ('pesan', 7), ('jadi', 7), ('baik', 7), ('aman', 7), ('tiba', 7)]


In [16]:
# --- Verifikasi manual sampel ---
# Fokus: (a) rating tidak sejalan dengan teks, (b) kasus campuran, (c) Unknown, (d) sampel acak.
rv = df[df["review_status"] == "Reviewed - With Text"].copy()
mismatch = rv[((rv["rating"] >= 4) & (rv["sentiment_label"] == "Negative")) |
              ((rv["rating"] <= 2) & (rv["sentiment_label"] == "Positive"))]
mixed    = rv[rv["sentiment_mixed"] == 1]
unknown  = rv[rv["sentiment_label"] == "Unknown"]
rand     = rv.sample(min(30, len(rv)), random_state=42) if len(rv) else rv

sample = pd.concat([mismatch.assign(reason="rating-teks tidak sejalan"),
                    mixed.assign(reason="pujian + keluhan"),
                    unknown.assign(reason="Unknown"),
                    rand.assign(reason="acak")]).drop_duplicates("order_id")
sample = sample[["order_id", "rating", "review_text", "sentiment_label", "reason"]].copy()
sample["manual_label"] = ""      # ISI dengan Positive/Neutral/Negative/Unknown setelah kamu baca teksnya
sample["catatan"] = ""
sample_path = OUT_DOCS / "sentiment_manual_sample.csv"
sample.to_csv(sample_path, index=False)
print(f"Sampel verifikasi: {len(sample)} baris -> {sample_path}")
print({"mismatch": len(mismatch), "mixed": len(mixed), "unknown": len(unknown), "random": len(rand)})
display(sample.head(15))

Sampel verifikasi: 84 baris -> C:\Users\hp\OneDrive\Dokumen\Future Maker Programe\Bootcamp_Data_Analyst\docs\capstone_swiftbite\sentiment_manual_sample.csv
{'mismatch': 30, 'mixed': 16, 'unknown': 27, 'random': 30}


,order_id,rating,review_text,sentiment_label,reason,manual_label,catatan
11,O00012,4.0,"Porsinya cukup banyak, tapi harganya terasa agak mahal.",Negative,rating-teks tidak sejalan,,
15,O00016,5.0,Packaging rapi dan tidak ada yang tumpah.,Negative,rating-teks tidak sejalan,,
32,O00033,1.0,Pengiriman biasa saja dan sesuai perkiraan.,Positive,rating-teks tidak sejalan,,
44,O00045,4.0,"Porsinya cukup banyak, tapi harganya terasa agak mahal.",Negative,rating-teks tidak sejalan,,
47,O00048,1.0,Makanannya masih fresh dan sesuai ekspektasi.,Positive,rating-teks tidak sejalan,,
48,O00049,5.0,"Porsinya cukup banyak, tapi harganya terasa agak mahal.",Negative,rating-teks tidak sejalan,,
52,O00053,1.0,Makanannya enak dan rasanya konsisten.,Positive,rating-teks tidak sejalan,,
75,O00076,4.0,Packaging kurang rapi dan ada yang tumpah.,Negative,rating-teks tidak sejalan,,
80,O00081,4.0,"Delivery cepat, tetapi packaging-nya kurang rapi.",Negative,rating-teks tidak sejalan,,
86,O00087,1.0,Harganya cukup worth it untuk kualitasnya.,Positive,rating-teks tidak sejalan,,


In [ ]:
# --- Setelah mengisi kolom manual_label di CSV sampel: hitung akurasi & terapkan koreksi ---
import os
if sample_path.exists():
    chk_s = pd.read_csv(sample_path, dtype=str, keep_default_na=False)
    done = chk_s[chk_s["manual_label"].str.strip() != ""]
    if len(done):
        agree = (done["manual_label"].str.strip().str.lower() == done["sentiment_label"].str.lower()).mean()
        print(f"Sampel yang sudah dilabel manual: {len(done)} | kesesuaian dengan rule-based: {agree:.1%}")
        EVIDENCE["sentiment_manual_n"] = int(len(done))
        EVIDENCE["sentiment_manual_agreement"] = round(float(agree), 4)
        # koreksi label berdasarkan hasil verifikasi manual
        VALID = {"Positive", "Neutral", "Negative", "Unknown"}
        fix = done[done["manual_label"].str.strip().isin(VALID)].set_index("order_id")["manual_label"].str.strip()
        idx = df["order_id"].isin(fix.index)
        df.loc[idx, "sentiment_label"] = df.loc[idx, "order_id"].map(fix)
        df.loc[idx, "sentiment_method"] = "manual_override"
        log(f"{int(idx.sum())} label sentimen dikoreksi manual dari sampel verifikasi.")
    else:
        print("Belum ada manual_label terisi. Isi dulu CSV sampel (minimal baris mismatch, mixed, Unknown), lalu jalankan ulang sel ini.")

## Tahap 5 - Insights (Penyusunan Temuan & Rekomendasi)

Tujuan: Merumuskan 3 hingga 5 temuan tervalidasi beserta rekomendasi bisnis yang aplikatif.   

Struktur Setiap Temuan: Harus memuat komponen Temuan, Evidence (Bukti Angka/Data), Interpretasi, Rekomendasi Tindakan, serta Keterbatasan analisis.

In [ ]:
# BQ1: tren per bulan
trend = (df[df["month"] != "Unknown"].groupby("month")
           .agg(total_orders=("order_id", "nunique"), net_payment=("net_payment", "sum"),
                reviewed=("has_review", "sum")).reset_index())
trend["review_coverage_pct"] = (100 * trend["reviewed"] / trend["total_orders"]).round(1)
trend["orders_mom_pct"] = (100 * trend["total_orders"].pct_change()).round(1)
display(trend)

In [ ]:
# BQ2: keterlambatan per restoran (sertakan N!)
MIN_N = 20   # ambang minimal delivered valid agar perbandingan layak (sesuaikan & jelaskan di report)
late_tbl = (df[df["delivered_valid"] == 1].groupby("restaurant")
              .agg(delivered_valid=("order_id", "nunique"), late_orders=("late_flag", "sum")).reset_index())
late_tbl["late_rate_pct"] = (100 * late_tbl["late_orders"] / late_tbl["delivered_valid"]).round(1)
late_tbl["n_memadai"] = late_tbl["delivered_valid"] >= MIN_N
overall_late = KPI_ALL["late_delivery_rate"]
print(f"Late Delivery Rate keseluruhan: {overall_late if overall_late is None else round(overall_late,1)}% "
      f"({KPI_ALL['_late_n']}/{KPI_ALL['_delivered_valid']})")
display(late_tbl.sort_values(["n_memadai", "late_rate_pct"], ascending=[False, False]).head(15))

# keterlambatan per bulan
late_m = (df[df["delivered_valid"] == 1].groupby("month")
            .agg(delivered_valid=("order_id", "nunique"), late_orders=("late_flag", "sum")).reset_index())
late_m["late_rate_pct"] = (100 * late_m["late_orders"] / late_m["delivered_valid"]).round(1)
display(late_m)

In [ ]:
# BQ3: customer experience
print("Review Coverage:", None if KPI_ALL["review_coverage"] is None else round(KPI_ALL["review_coverage"], 1), "%")
sent_tbl = df["sentiment_label"].value_counts(dropna=True).reindex(["Positive", "Neutral", "Negative", "Unknown", "No Text"]).fillna(0).astype(int)
print("\nDistribusi sentimen (hanya order yang punya record review):")
print(sent_tbl)
print("\nStatus order:\n", df["review_status"].value_counts())

print("\nCrosstab rating x sentimen (cek rating vs teks):")
display(pd.crosstab(df["rating"], df["sentiment_label"], dropna=True))

by_rest = (df[df["has_review"] == 1].groupby("restaurant")
             .agg(reviews=("order_id", "nunique"), avg_rating=("rating", "mean"),
                  negative=("sentiment_label", lambda s: (s == "Negative").sum())).reset_index())
by_rest["negative_pct"] = (100 * by_rest["negative"] / by_rest["reviews"]).round(1)
by_rest["avg_rating"] = by_rest["avg_rating"].round(2)
display(by_rest.sort_values("reviews", ascending=False).head(15))

# apakah order terlambat cenderung dapat rating lebih rendah? (indikasi, BUKAN sebab-akibat)
x = df[(df["delivered_valid"] == 1) & df["rating"].notna()]
if len(x):
    display(x.groupby("late_flag")["rating"].agg(["count", "mean"]).rename(index={0.0: "tepat waktu", 1.0: "terlambat"}).round(2))

In [ ]:
# Keluhan yang sering muncul pada review Negative (bantu menulis insight)
neg_txt = df.loc[df["sentiment_label"] == "Negative", "review_text"].dropna()
c = Counter(tok for t in neg_txt for tok in TOKEN_RE.findall(str(t).lower()) if tok not in NEGATORS and len(tok) > 3)
print(f"Review Negative: {len(neg_txt)}")
print("Kata dominan:", c.most_common(20))

## Tahap 6 - Dashboard & Report (Produksi Output Visual & Laporan)

Tujuan: Membangun sarana penyampaian hasil analisis agar mudah dipahami oleh stakeholder atau mentor.   

Output yang Dibuat:
1. Dashboard HTML Interaktif: Minimal memiliki 4 KPI, 3 grafik, dan 2 filter yang berfungsi secara dinamis serta dapat dibuka langsung di browser tanpa environment koding khusus.   
2. Analysis Report (PPTX/PDF): Laporan terstruktur yang menjelaskan konteks proyek, metode, validasi, temuan, hingga rekomendasi bisnis.

In [ ]:
def _clean(v):
    if v is None or (isinstance(v, float) and np.isnan(v)) or v is pd.NA:
        return None
    return v

rec_df = pd.DataFrame({
    "m":  df["month"],
    "r":  df["restaurant"].astype(str),
    "z":  df["customer_zone"].astype(str),
    "np": df["net_payment"].round(2),
    "hr": df["has_review"].astype(int),
    "dv": df["delivered_valid"].astype(int),
    "lt": df["late_flag"].fillna(0).astype(int),
    "s":  df["sentiment_label"].where(df["has_review"] == 1, "No Review").astype(str),
    "rt": df["rating"],
    "cs": df["final_payment_status"].where(df["order_status"] == "CANCELLED", None),
})
records = [{k: _clean(v) for k, v in row.items()} for row in rec_df.to_dict("records")]
# order_id sengaja TIDAK dimasukkan ke dashboard (tidak dibutuhkan, mengurangi risiko data)

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>SwiftBite Insight Hub __VARIANT__</title>
<style>
:root {
  --primary: #E8590C;
  --primary-hover: #C2410C;
  --primary-light: #F59F42;
  --primary-subtle: #FFE8D6;
  --text-dark: #2B1A10;
  --text-muted: #7C6F64;
  --bg-cream: #FFF8F0;
  --card-bg: #FFFFFF;
  --success: #1B9E6B;
  --danger: #DC2626;
  --warning: #D97706;
  --info: #2563EB;
  --border-light: rgba(232, 89, 12, 0.12);
  --border-card-top: #FFF1E6;
  --sidebar-bg: #FFE8D6;
  --sidebar-border: #F5D3BC;
  --shadow-sm: 0 1px 2px rgba(120, 60, 10, 0.08);
  --shadow-md: 0 8px 24px rgba(120, 60, 10, 0.12);
  --shadow-hover: 0 12px 32px rgba(120, 60, 10, 0.16);
}

*, *::before, *::after {
  box-sizing: border-box;
  margin: 0;
  padding: 0;
}

body {
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
  background-color: var(--bg-cream);
  color: var(--text-dark);
  line-height: 1.5;
  -webkit-font-smoothing: antialiased;
}

/* APP LAYOUT: FULL WIDTH */
.app-layout {
  display: flex;
  min-height: 100vh;
  width: 100%;
  background-color: var(--bg-cream);
}

/* SIDEBAR KIRI (200px, Sticky setinggi layar, Ringkasan Cepat) */
.sidebar {
  width: 200px;
  min-width: 200px;
  max-width: 200px;
  background: var(--sidebar-bg);
  border-right: 1.5px solid var(--sidebar-border);
  box-shadow: 2px 0 16px rgba(120, 60, 10, 0.06);
  display: flex;
  flex-direction: column;
  justify-content: space-between;
  position: sticky;
  top: 0;
  height: 100vh;
  z-index: 200;
  padding: 16px 10px;
  overflow-y: auto;
}

.sidebar-top {
  display: flex;
  flex-direction: column;
  gap: 12px;
}

.sidebar-brand {
  padding: 4px 8px 12px 8px;
  border-bottom: 1.5px solid var(--sidebar-border);
}

.brand-title {
  font-size: 19px;
  font-weight: 800;
  color: var(--primary);
  letter-spacing: -0.4px;
  display: flex;
  align-items: center;
  gap: 6px;
}

.brand-subtitle {
  font-size: 11px;
  font-weight: 700;
  color: var(--text-muted);
  text-transform: uppercase;
  letter-spacing: 0.6px;
  margin-top: 1px;
}

.sidebar-menu {
  list-style: none;
  display: flex;
  flex-direction: column;
  gap: 4px;
}

.menu-btn {
  width: 100%;
  min-height: 48px;
  display: flex;
  align-items: center;
  gap: 8px;
  padding: 0 12px;
  border: none;
  border-radius: 12px;
  background: transparent;
  color: var(--text-dark);
  font-size: 15px;
  font-weight: 600;
  cursor: pointer;
  text-align: left;
  transition: all 0.18s ease;
}

.menu-icon {
  font-size: 17px;
  display: inline-flex;
  align-items: center;
  justify-content: center;
  width: 22px;
  flex-shrink: 0;
}

.menu-btn:hover {
  background: rgba(232, 89, 12, 0.10);
  color: var(--primary);
  transform: translateX(2px);
}

.menu-btn.active {
  background: var(--primary);
  color: #FFFFFF;
  font-weight: 700;
  box-shadow: 0 4px 12px rgba(232, 89, 12, 0.35);
  transform: translateY(-1px);
}

.menu-btn:active {
  transform: translateY(1px);
}

.sidebar-summary-card {
  background: #FFFFFF;
  border: 1.5px solid var(--sidebar-border);
  border-radius: 12px;
  padding: 10px 12px;
  box-shadow: var(--shadow-sm);
  display: flex;
  flex-direction: column;
  gap: 6px;
  margin-top: 10px;
}

.sidebar-summary-title {
  font-size: 11px;
  font-weight: 800;
  color: var(--primary);
  text-transform: uppercase;
  letter-spacing: 0.5px;
  margin-bottom: 2px;
}

.sidebar-summary-item {
  display: flex;
  justify-content: space-between;
  align-items: baseline;
  border-bottom: 1px dashed rgba(232, 89, 12, 0.12);
  padding-bottom: 4px;
}

.sidebar-summary-item:last-child {
  border-bottom: none;
  padding-bottom: 0;
}

.ss-lbl {
  font-size: 11px;
  font-weight: 600;
  color: var(--text-muted);
}

.ss-val {
  font-size: 14px;
  font-weight: 800;
  color: var(--text-dark);
}

.sidebar-bottom {
  display: flex;
  flex-direction: column;
  gap: 8px;
  padding-top: 10px;
}

.btn-sidebar-explain {
  width: 100%;
  min-height: 42px;
  padding: 0 10px;
  background: #FFFFFF;
  border: 1.5px solid var(--sidebar-border);
  border-radius: 10px;
  font-size: 13px;
  font-weight: 700;
  color: var(--primary);
  cursor: pointer;
  display: flex;
  align-items: center;
  justify-content: center;
  gap: 6px;
  box-shadow: var(--shadow-sm);
  transition: all 0.2s ease;
}

.btn-sidebar-explain:hover, .btn-sidebar-explain.active {
  background: var(--primary);
  color: #FFFFFF;
  border-color: var(--primary);
  box-shadow: 0 4px 12px rgba(232, 89, 12, 0.25);
}

.btn-sidebar-explain:active {
  transform: translateY(1px);
}

.sidebar-variant-tag {
  font-size: 11px;
  color: var(--text-muted);
  text-align: center;
  font-weight: 600;
}

/* MAIN CONTENT WRAPPER: FULL WIDTH */
.main-wrapper {
  flex: 1;
  width: calc(100% - 200px);
  max-width: none !important;
  margin: 0 !important;
  padding: 22px 28px 56px 28px;
}

/* HEADER & GREETING */
.greeting-section {
  margin-bottom: 18px;
}

.page-title {
  font-size: 20px;
  font-weight: 800;
  color: var(--primary);
  letter-spacing: -0.4px;
}

.page-subtitle {
  font-size: 13px;
  color: var(--text-muted);
  margin-top: 1px;
}

.greeting-headline {
  font-size: 29px;
  font-weight: 800;
  color: var(--text-dark);
  letter-spacing: -0.7px;
  margin-top: 8px;
  display: flex;
  align-items: center;
  gap: 8px;
}

.greeting-sub {
  font-size: 13px;
  color: var(--text-muted);
  margin-top: 3px;
}

/* 1. BAR FILTER: PROMINENT ORANGE GRADIENT BANNER (#E8590C ke #F59F42, OPAQUE, STICKY) */
.filter-banner-card {
  background: linear-gradient(135deg, #E8590C 0%, #F59F42 100%) !important;
  border-radius: 16px;
  box-shadow: 0 8px 24px rgba(232, 89, 12, 0.28);
  padding: 18px 22px;
  margin-bottom: 24px;
  position: sticky;
  top: 12px;
  z-index: 500;
  display: flex;
  flex-direction: column;
  gap: 14px;
  color: #FFFFFF;
  transition: all 0.25s ease;
}

.filter-banner-card.is-sticky {
  padding: 8px 18px;
  gap: 8px;
  box-shadow: 0 10px 30px rgba(120, 60, 10, 0.35);
}

.filter-banner-card.is-sticky .filter-banner-desc {
  display: none;
}

.filter-banner-card.is-sticky .filter-field-group {
  gap: 3px;
}

.filter-banner-card.is-sticky .filter-field-select {
  height: 40px;
  font-size: 14.5px;
}

.filter-banner-top {
  display: flex;
  align-items: center;
  justify-content: space-between;
  flex-wrap: wrap;
  gap: 12px;
}

.filter-banner-titlebox {
  display: flex;
  flex-direction: column;
  gap: 2px;
}

.filter-banner-title {
  font-size: 18px;
  font-weight: 800;
  color: #FFFFFF;
  display: flex;
  align-items: center;
  gap: 8px;
  letter-spacing: -0.3px;
}

.filter-banner-desc {
  font-size: 12.5px;
  color: rgba(255, 255, 255, 0.92);
  font-weight: 500;
}

.filter-banner-top-right {
  display: flex;
  align-items: center;
  gap: 14px;
}

/* Penghitung hidup pesanan */
.filter-live-counter {
  background: rgba(0, 0, 0, 0.18);
  border: 1px solid rgba(255, 255, 255, 0.3);
  padding: 8px 14px;
  border-radius: 10px;
  font-size: 13.5px;
  font-weight: 700;
  color: #FFFFFF;
  white-space: nowrap;
}

.filter-live-counter .counter-num {
  font-size: 15px;
  font-weight: 800;
  color: #FFD166;
}

.btn-reset-white {
  min-height: 44px;
  padding: 0 16px;
  background: #FFFFFF;
  border: 1.5px solid #FFFFFF;
  border-radius: 10px;
  font-size: 14px;
  font-weight: 800;
  color: var(--primary);
  cursor: pointer;
  display: inline-flex;
  align-items: center;
  gap: 6px;
  box-shadow: 0 2px 8px rgba(0,0,0,0.12);
  transition: all 0.2s ease;
}

.btn-reset-white:not(:disabled):hover {
  background: #FFF8F0;
  color: var(--primary-hover);
  transform: translateY(-1px);
  box-shadow: 0 4px 12px rgba(0,0,0,0.18);
}

.btn-reset-white:disabled {
  opacity: 0.5;
  cursor: not-allowed;
  background: rgba(255, 255, 255, 0.4);
  color: #FFFFFF;
  border-color: transparent;
  box-shadow: none;
}

.btn-reset-white:active {
  transform: translateY(1px);
}

/* Dropdown controls with labels on top */
.filter-dropdowns-grid {
  display: grid;
  grid-template-columns: 1fr 1fr 1fr;
  gap: 14px;
  width: 100%;
}

.filter-field-group {
  display: flex;
  flex-direction: column;
  gap: 6px;
}

.filter-field-label {
  font-size: 13px;
  font-weight: 700;
  color: #FFFFFF;
  display: flex;
  align-items: center;
  justify-content: space-between;
}

.filter-active-dot {
  width: 8px;
  height: 8px;
  border-radius: 50%;
  background: #FFD166;
  box-shadow: 0 0 8px #FFD166;
  display: none;
}

.filter-field-group.is-active .filter-active-dot {
  display: inline-block;
}

.filter-field-select {
  height: 48px;
  font-size: 16px;
  font-weight: 600;
  border-radius: 10px;
  padding: 0 14px;
  border: 2px solid transparent;
  background: #FFFFFF;
  color: var(--text-dark);
  cursor: pointer;
  transition: all 0.2s ease;
  outline: none;
  width: 100%;
}

.filter-field-select:focus {
  border-color: #FFD166;
  box-shadow: 0 0 0 3px rgba(255, 209, 102, 0.4);
}

/* Dropdown aktif diberi garis tebal kuning #FFD166 */
.filter-field-select.active-filter {
  border: 2.5px solid #FFD166 !important;
  background: #FFFDF5 !important;
  font-weight: 700;
}

/* Active filter chips */
.filter-chips-bar {
  display: flex;
  align-items: center;
  gap: 8px;
  flex-wrap: wrap;
  padding-top: 4px;
  border-top: 1px dashed rgba(255, 255, 255, 0.35);
}

.filter-chip-item {
  display: inline-flex;
  align-items: center;
  gap: 6px;
  background: rgba(255, 255, 255, 0.22);
  border: 1px solid rgba(255, 255, 255, 0.45);
  color: #FFFFFF;
  padding: 4px 10px;
  border-radius: 20px;
  font-size: 12.5px;
  font-weight: 700;
  backdrop-filter: blur(4px);
}

.filter-chip-remove {
  background: transparent;
  border: none;
  color: #FFFFFF;
  font-size: 13px;
  font-weight: 800;
  cursor: pointer;
  padding: 0 2px;
  line-height: 1;
}

.filter-chip-remove:hover {
  color: #FFD166;
}

.filter-chip-none {
  font-size: 12px;
  color: rgba(255, 255, 255, 0.85);
  font-style: italic;
}

/* SCROLL-MARGIN-TOP SO HEADINGS ARE NEVER COVERED */
.card-wrapper, .kpi-row, .middle-row, .bottom-row, .bq04-row {
  scroll-margin-top: 140px;
}

/* FLASH/BLINK HIGHLIGHT ON FILTER CHANGE (~0.6s) */
@keyframes cardPulse {
  0% { box-shadow: 0 0 0 0 rgba(232, 89, 12, 0); background-color: #FFFFFF; }
  35% { box-shadow: 0 0 0 5px rgba(245, 159, 66, 0.45); background-color: #FFF3E6; }
  100% { box-shadow: var(--shadow-sm), var(--shadow-md); background-color: #FFFFFF; }
}

.card-highlight .card-face {
  animation: cardPulse 0.6s ease-out;
}

/* 3D FLASHCARD STRUCTURE (Ukuran konsisten, tidak meluap) */
.card-wrapper {
  perspective: 1000px;
  position: relative;
  border-radius: 16px;
  transition: transform 0.25s cubic-bezier(0.4, 0, 0.2, 1);
}

.card-wrapper:not(.flipped):hover {
  transform: translateY(-4px);
}

.card-inner {
  position: relative;
  width: 100%;
  height: 100%;
  border-radius: 16px;
  transform-style: preserve-3d;
  transition: transform 0.4s cubic-bezier(0.4, 0, 0.2, 1);
}

.card-wrapper.flipped .card-inner {
  transform: rotateY(180deg);
}

.card-face {
  width: 100%;
  height: 100%;
  background: linear-gradient(180deg, #FFFFFF 0%, #FFF8F0 100%);
  border: 1px solid var(--border-light);
  border-top: 1.5px solid var(--border-card-top);
  border-radius: 16px;
  box-shadow: var(--shadow-sm), var(--shadow-md);
  backface-visibility: hidden;
  -webkit-backface-visibility: hidden;
  display: flex;
  flex-direction: column;
  position: relative;
  transition: box-shadow 0.25s ease;
}

.card-wrapper:hover .card-face {
  box-shadow: 0 4px 8px rgba(120, 60, 10, 0.10), var(--shadow-hover);
}

/* FLASHCARD DISEDERHANAKAN: PERSIS SETINGGI SISI DEPAN */
.card-back {
  position: absolute;
  top: 0; left: 0; right: 0; bottom: 0;
  transform: rotateY(180deg);
  background: #FFFDFC;
  padding: 16px 18px;
  display: flex;
  flex-direction: column;
  justify-content: space-between;
  border-radius: 16px;
  box-sizing: border-box;
}

.btn-flip {
  position: absolute;
  top: 12px;
  right: 12px;
  width: 24px;
  height: 24px;
  border-radius: 50%;
  background: rgba(232, 89, 12, 0.12);
  color: var(--primary);
  border: 1px solid rgba(232, 89, 12, 0.25);
  font-size: 12px;
  font-weight: 800;
  font-family: Georgia, serif;
  cursor: pointer;
  display: flex;
  align-items: center;
  justify-content: center;
  z-index: 10;
  transition: all 0.2s ease;
}

.btn-flip:hover {
  background: var(--primary);
  color: #FFFFFF;
  transform: scale(1.1);
  box-shadow: 0 2px 6px rgba(232, 89, 12, 0.3);
}

.btn-flip:active {
  transform: scale(0.95);
}

.back-header {
  display: flex;
  align-items: center;
  gap: 8px;
  padding-right: 28px;
  border-bottom: 1.5px solid #F5E5D8;
  padding-bottom: 6px;
}

.back-header h4 {
  font-size: 13.5px;
  font-weight: 800;
  color: var(--primary);
}

.back-meaning {
  font-size: 14px;
  line-height: 1.45;
  color: var(--text-dark);
  margin-top: 6px;
  font-weight: 500;
}

.back-kpi-sub {
  margin-top: auto;
  padding-top: 6px;
  border-top: 1px dashed rgba(232, 89, 12, 0.25);
  font-size: 12px;
  font-weight: 700;
  color: var(--primary);
  font-family: ui-monospace, monospace;
}

/* BARIS KPI (5 KOLOM SAMA LEBAR, PENUH) */
.kpi-row {
  display: grid;
  grid-template-columns: repeat(5, 1fr);
  gap: 16px;
  margin-bottom: 24px;
  width: 100%;
}

.kpi-card-wrap {
  min-height: 148px;
  height: 100%;
}

.kpi-card-front {
  padding: 16px 16px 14px 16px;
  justify-content: space-between;
}

.kpi-top {
  display: flex;
  align-items: center;
  gap: 10px;
  padding-right: 26px;
}

.kpi-icon-box {
  width: 38px;
  height: 38px;
  min-width: 38px;
  border-radius: 12px;
  background: linear-gradient(135deg, #FF9E66 0%, #E8590C 100%);
  box-shadow: 0 4px 12px rgba(232, 89, 12, 0.28);
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 18px;
  color: #FFFFFF;
  position: relative;
  overflow: hidden;
}

.kpi-icon-box::after {
  content: "";
  position: absolute;
  top: 0; left: 0; right: 0; height: 50%;
  background: linear-gradient(180deg, rgba(255,255,255,0.35) 0%, rgba(255,255,255,0) 100%);
  border-radius: 12px 12px 0 0;
  pointer-events: none;
}

.kpi-title {
  font-size: 13px;
  font-weight: 700;
  color: var(--text-muted);
  line-height: 1.2;
}

.kpi-middle {
  display: flex;
  align-items: flex-end;
  justify-content: space-between;
  margin-top: 6px;
  gap: 6px;
}

.kpi-value {
  font-size: 23px;
  font-weight: 800;
  color: var(--text-dark);
  letter-spacing: -0.6px;
  line-height: 1.1;
}

.kpi-sparkline {
  width: 60px;
  height: 26px;
  overflow: visible;
  flex-shrink: 0;
}

.kpi-bottom {
  font-size: 11.5px;
  font-weight: 600;
  color: var(--text-muted);
  margin-top: 6px;
  display: flex;
  align-items: center;
  gap: 4px;
}

.trend-badge {
  display: inline-flex;
  align-items: center;
  gap: 3px;
  padding: 2px 7px;
  border-radius: 6px;
  font-size: 11px;
  font-weight: 700;
}

.trend-up {
  background: rgba(27, 158, 107, 0.12);
  color: var(--success);
}

.trend-down {
  background: rgba(220, 38, 38, 0.12);
  color: var(--danger);
}

.trend-neutral {
  background: rgba(124, 111, 100, 0.12);
  color: var(--text-muted);
}

/* BARIS TENGAH (2fr : 1fr) */
.middle-row {
  display: grid;
  grid-template-columns: 2fr 1fr;
  gap: 20px;
  margin-bottom: 24px;
  width: 100%;
}

.bq04-row {
  display: grid;
  grid-template-columns: 1fr;
  gap: 20px;
  margin-bottom: 24px;
  width: 100%;
}

/* BARIS BAWAH (3 KOLOM SAMA LEBAR) */
.bottom-row {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 20px;
  margin-bottom: 24px;
  width: 100%;
}

.chart-card-front, .bottom-card-front {
  padding: 18px 20px;
}

.card-header-bar {
  display: flex;
  align-items: center;
  justify-content: space-between;
  margin-bottom: 12px;
  flex-wrap: wrap;
  gap: 10px;
  padding-right: 32px;
}

.card-heading {
  font-size: 16px;
  font-weight: 800;
  color: var(--text-dark);
  letter-spacing: -0.3px;
  display: flex;
  align-items: center;
  gap: 8px;
}

/* TABS PIL DI KANAN ATAS */
.tabs-group {
  display: inline-flex;
  background: #F3E5D8;
  border-radius: 20px;
  padding: 3px;
  gap: 2px;
}

.tab-btn {
  border: none;
  background: transparent;
  padding: 5px 14px;
  border-radius: 16px;
  font-size: 12px;
  font-weight: 700;
  color: var(--text-muted);
  cursor: pointer;
  transition: all 0.2s ease;
}

.tab-btn.active {
  background: #FFFFFF;
  color: var(--primary);
  box-shadow: 0 2px 6px rgba(120, 60, 10, 0.12);
}

.tab-btn:active {
  transform: translateY(1px);
}

/* GRAFIK TREN: DUA PANEL BERDAMPINGAN */
.trend-panels-grid {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 14px;
  width: 100%;
}

.trend-subpanel {
  background: #FFF8F0;
  border: 1px solid #F5D3BC;
  border-radius: 14px;
  padding: 12px 14px;
  display: flex;
  flex-direction: column;
  gap: 6px;
}

.panel-header {
  display: flex;
  align-items: baseline;
  justify-content: space-between;
  border-bottom: 1px solid rgba(232, 89, 12, 0.12);
  padding-bottom: 6px;
}

.panel-title {
  font-size: 12.5px;
  font-weight: 700;
  color: var(--text-muted);
}

.panel-kpi-box {
  display: flex;
  align-items: baseline;
  gap: 5px;
}

.panel-big-num {
  font-size: 20px;
  font-weight: 800;
  color: var(--text-dark);
  letter-spacing: -0.4px;
  line-height: 1;
}

.panel-sub-lbl {
  font-size: 10.5px;
  font-weight: 600;
  color: var(--text-muted);
}

.panel-chart-wrap {
  width: 100%;
  height: 185px;
}

.panel-chart-wrap svg {
  width: 100%;
  height: 100%;
  overflow: visible;
  display: block;
}

/* STRIP RINGKASAN 4 KOLOM DI BAWAH GRAFIK TREN */
.trend-summary-strip {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 10px;
  margin-top: 12px;
  background: #FFF1E6;
  border: 1px solid #F5D3BC;
  border-radius: 12px;
  padding: 10px 14px;
}

.strip-col {
  display: flex;
  align-items: center;
  gap: 8px;
}

.strip-icon {
  font-size: 17px;
}

.strip-info {
  display: flex;
  flex-direction: column;
}

.strip-lbl {
  font-size: 10.5px;
  font-weight: 700;
  color: var(--text-muted);
  text-transform: uppercase;
  letter-spacing: 0.3px;
}

.strip-val {
  font-size: 12px;
  font-weight: 800;
  color: var(--primary);
  white-space: nowrap;
}

/* 3. STRIP "TEMUAN UTAMA" DAN "MAKNA BISNIS" PADA SETIAP KARTU GRAFIK */
.card-insight-strip {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 12px;
  margin-top: 14px;
}

.strip-box {
  border-radius: 10px;
  padding: 10px 14px;
  display: flex;
  flex-direction: column;
  gap: 4px;
}

.strip-box-findings {
  background: #FFF3C4;
  border: 1px solid #FFE69C;
}

.strip-box-business {
  background: #FFE8D6;
  border: 1px solid #F5D3BC;
}

.strip-header {
  display: flex;
  align-items: center;
  gap: 6px;
}

.strip-header-icon {
  font-size: 13.5px;
}

.strip-header-title {
  font-size: 11.5px;
  font-weight: 800;
  text-transform: uppercase;
  letter-spacing: 0.4px;
}

.strip-box-findings .strip-header-title {
  color: #7A5B04;
}

.strip-box-business .strip-header-title {
  color: #C2410C;
}

.strip-text {
  font-size: 14px;
  line-height: 1.45;
  color: #2B1A10;
  font-weight: 500;
}

/* DONUT SENTIMEN REVIEW */
.donut-layout {
  display: flex;
  flex-direction: column;
  gap: 12px;
  height: 100%;
  justify-content: space-between;
}

.donut-visual-row {
  display: flex;
  align-items: center;
  justify-content: space-around;
  gap: 12px;
  flex: 1;
}

.donut-svg-wrap {
  position: relative;
  width: 150px;
  height: 150px;
  filter: drop-shadow(0 4px 8px rgba(120,60,10,0.10));
}

.donut-center-text {
  position: absolute;
  top: 50%;
  left: 50%;
  transform: translate(-50%, -50%);
  text-align: center;
  pointer-events: none;
}

.donut-center-num {
  font-size: 24px;
  font-weight: 800;
  color: var(--text-dark);
  line-height: 1;
}

.donut-center-lbl {
  font-size: 10.5px;
  font-weight: 700;
  color: var(--text-muted);
  text-transform: uppercase;
  margin-top: 1px;
}

.donut-legend {
  display: flex;
  flex-direction: column;
  gap: 5px;
}

.donut-leg-row {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 10px;
  font-size: 12px;
}

.donut-leg-key {
  display: flex;
  align-items: center;
  gap: 6px;
  font-weight: 600;
  color: var(--text-dark);
}

.donut-dot {
  width: 9px;
  height: 9px;
  border-radius: 50%;
}

.donut-leg-val {
  font-weight: 700;
}

/* BQ-04: KETERLAMBATAN & RATING */
.bq4-body {
  display: flex;
  flex-direction: column;
  gap: 12px;
}

.bq4-grid {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 14px;
}

.bq4-col {
  border-radius: 12px;
  padding: 12px 14px;
  display: flex;
  flex-direction: column;
  gap: 8px;
}

.bq4-col-late {
  background: rgba(220, 38, 38, 0.05);
  border: 1.5px solid rgba(220, 38, 38, 0.2);
}

.bq4-col-ontime {
  background: rgba(27, 158, 107, 0.05);
  border: 1.5px solid rgba(27, 158, 107, 0.2);
}

.bq4-col-header {
  display: flex;
  align-items: center;
  justify-content: space-between;
}

.bq4-badge {
  font-size: 11.5px;
  font-weight: 800;
  padding: 2px 7px;
  border-radius: 6px;
}

.bq4-badge-late {
  background: rgba(220, 38, 38, 0.12);
  color: var(--danger);
}

.bq4-badge-ontime {
  background: rgba(27, 158, 107, 0.12);
  color: var(--success);
}

.bq4-stat-main {
  display: flex;
  align-items: baseline;
  gap: 8px;
}

.bq4-big-num {
  font-size: 30px;
  font-weight: 800;
  letter-spacing: -0.7px;
  line-height: 1;
}

.bq4-col-late .bq4-big-num { color: var(--danger); }
.bq4-col-ontime .bq4-big-num { color: var(--success); }

.bq4-sub-metric {
  font-size: 11.5px;
  font-weight: 700;
  color: var(--text-muted);
}

.bq4-details {
  display: flex;
  flex-direction: column;
  gap: 3px;
  font-size: 12px;
  color: var(--text-dark);
  border-top: 1px dashed rgba(0,0,0,0.08);
  padding-top: 6px;
}

.bq4-row {
  display: flex;
  justify-content: space-between;
}

.text-danger { color: var(--danger); }
.text-success { color: var(--success); }

/* BQ-04 BAR CHART (0 to 5 SCALE) */
.bq4-chart-wrap {
  background: #FFFFFF;
  border: 1px solid #EADCD0;
  border-radius: 12px;
  padding: 10px 16px;
  display: flex;
  flex-direction: column;
  gap: 6px;
}

.bq4-chart-title {
  font-size: 12px;
  font-weight: 700;
  color: var(--text-muted);
}

.bq4-bar-row {
  display: flex;
  align-items: center;
  gap: 10px;
}

.bq4-bar-label {
  width: 85px;
  font-size: 12px;
  font-weight: 700;
  color: var(--text-dark);
}

.bq4-bar-track {
  flex: 1;
  height: 20px;
  background: #F3E5D8;
  border-radius: 6px;
  overflow: hidden;
  position: relative;
}

.bq4-bar-fill {
  height: 100%;
  border-radius: 6px;
  display: flex;
  align-items: center;
  justify-content: flex-end;
  padding-right: 8px;
  transition: width 0.5s ease;
}

.bq4-bar-late {
  background: linear-gradient(90deg, #F87171, #DC2626);
}

.bq4-bar-ontime {
  background: linear-gradient(90deg, #34D399, #1B9E6B);
}

.bq4-bar-val {
  font-size: 11.5px;
  font-weight: 800;
  color: #FFFFFF;
}

.bq4-axis {
  display: flex;
  justify-content: space-between;
  padding-left: 95px;
  font-size: 10px;
  font-weight: 700;
  color: var(--text-muted);
}

.bq4-note {
  font-size: 11.5px;
  font-weight: 600;
  color: var(--text-muted);
  font-style: italic;
  padding-left: 2px;
}

/* LATE RATE PER ZONA */
.zone-rank-list {
  display: flex;
  flex-direction: column;
  gap: 8px;
}

.zone-rank-item {
  display: flex;
  flex-direction: column;
  gap: 3px;
}

.zone-item-top {
  display: flex;
  justify-content: space-between;
  font-size: 12.5px;
  font-weight: 700;
}

.zone-name {
  color: var(--text-dark);
  display: flex;
  align-items: center;
  gap: 5px;
}

.zone-badge-num {
  width: 17px;
  height: 17px;
  background: #EAD7C7;
  color: var(--text-dark);
  font-size: 10.5px;
  border-radius: 50%;
  display: inline-flex;
  align-items: center;
  justify-content: center;
}

.zone-item-top.accent .zone-badge-num {
  background: var(--primary);
  color: #FFFFFF;
}

.zone-rate-text {
  color: var(--text-muted);
}

.zone-item-top.accent .zone-rate-text {
  color: var(--primary);
  font-weight: 800;
}

.zone-bar-track {
  width: 100%;
  height: 7px;
  background: #F3E5D8;
  border-radius: 4px;
  overflow: hidden;
}

.zone-bar-fill {
  height: 100%;
  border-radius: 4px;
  background: #C2A28C;
  transition: width 0.4s ease;
}

.zone-bar-fill.accent {
  background: linear-gradient(90deg, #F59F42, #E8590C);
}

/* 2. STATUS PESANAN (GAUGE SETENGAH LINGKARAN PENUH) */
.gauge-layout {
  display: flex;
  flex-direction: column;
  align-items: center;
  gap: 10px;
  height: 100%;
  justify-content: space-between;
}

.gauge-svg-wrap {
  position: relative;
  width: 190px;
  height: 110px;
  filter: drop-shadow(0 4px 8px rgba(120,60,10,0.10));
}

.gauge-center-info {
  position: absolute;
  bottom: 8px;
  left: 50%;
  transform: translateX(-50%);
  text-align: center;
}

.gauge-pct-text {
  font-size: 24px;
  font-weight: 800;
  color: var(--text-dark);
  line-height: 1;
}

.gauge-sub-text {
  font-size: 11px;
  font-weight: 700;
  color: var(--text-muted);
  text-transform: uppercase;
}


/* 2. ORDER STATUS COLUMNS WITH COLORED DOTS & PERCENTAGES */
.status-cols-row {
  display: flex;
  justify-content: space-between;
  gap: 8px;
  width: 100%;
  margin-top: 10px;
  padding: 8px 10px;
  background: var(--bg-cream);
  border-radius: 10px;
  border: 1px solid var(--border-light);
}

.status-col {
  display: flex;
  flex-direction: column;
  align-items: center;
  gap: 2px;
  flex: 1;
}

.status-col-top {
  display: flex;
  align-items: center;
  gap: 5px;
  font-size: 13px;
  color: var(--text-dark);
}

.status-col-dot {
  width: 8px;
  height: 8px;
  border-radius: 50%;
  display: inline-block;
  flex-shrink: 0;
}

.status-col-label {
  font-size: 12.5px;
  font-weight: 600;
  color: var(--text-dark);
}

.status-col-label strong {
  font-weight: 800;
}

.status-col-pct {
  font-size: 11.5px;
  font-weight: 700;
  color: var(--text-muted);
}

.status-chips-row {
  display: flex;
  gap: 6px;
  width: 100%;
}

.status-chip {
  flex: 1;
  background: #FFF1E6;
  border-radius: 8px;
  padding: 6px 8px;
  display: flex;
  flex-direction: column;
  align-items: center;
  box-shadow: var(--shadow-sm);
}

.status-chip strong {
  font-size: 14px;
  font-weight: 800;
  color: var(--text-dark);
}

.status-chip span {
  font-size: 10px;
  font-weight: 700;
  color: var(--text-muted);
  text-transform: uppercase;
}

/* BUSINESS INSIGHTS (4 POIN OTOMATIS) */
.insights-list {
  display: flex;
  flex-direction: column;
  gap: 8px;
}

.insight-item {
  display: flex;
  align-items: flex-start;
  gap: 8px;
  font-size: 12px;
  line-height: 1.45;
  color: var(--text-dark);
  background: #FFFDF9;
  border: 1px solid #F5E5D8;
  border-radius: 10px;
  padding: 8px 10px;
}

.insight-bullet {
  font-size: 11.5px;
  font-weight: 800;
  color: var(--primary);
  background: var(--primary-subtle);
  width: 20px;
  height: 20px;
  min-width: 20px;
  border-radius: 50%;
  display: flex;
  align-items: center;
  justify-content: center;
}

/* EXPANDED FULL-WIDTH HELPER ON NAVIGATION */
.section-full-width {
  grid-column: 1 / -1 !important;
}

/* RESPONSIVE DESIGN */
@media (max-width: 1140px) {
  .kpi-row { grid-template-columns: repeat(3, 1fr); }
  .middle-row { grid-template-columns: 1fr; }
  .bottom-row { grid-template-columns: 1fr; }
  .card-insight-strip { grid-template-columns: 1fr; }
}

@media (max-width: 768px) {
  .app-layout { flex-direction: column; }
  .sidebar {
    width: 100%;
    min-width: 100%;
    max-width: 100%;
    height: auto;
    position: relative;
    padding: 12px 14px;
    border-right: none;
    border-bottom: 1.5px solid var(--sidebar-border);
  }
  .sidebar-top { flex-direction: row; align-items: center; justify-content: space-between; gap: 8px; }
  .sidebar-brand { border-bottom: none; padding: 0; }
  .sidebar-menu {
    flex-direction: row;
    overflow-x: auto;
    padding-bottom: 4px;
    width: 100%;
  }
  .menu-btn { white-space: nowrap; min-height: 40px; padding: 0 10px; font-size: 14px; }
  .sidebar-summary-card, .sidebar-bottom { display: none; }
  .main-wrapper { width: 100%; padding: 16px 14px; }
  .filter-banner-card { position: relative !important; top: 0 !important; }
  .filter-dropdowns-grid { grid-template-columns: 1fr; }
  .filter-banner-top-right { width: 100%; justify-content: space-between; }
  .kpi-row { grid-template-columns: 1fr 1fr; }
  .bq4-grid { grid-template-columns: 1fr; }
  .trend-panels-grid { grid-template-columns: 1fr; }
  .trend-summary-strip { grid-template-columns: 1fr 1fr; }
  .card-insight-strip { grid-template-columns: 1fr; }
}

@media (max-width: 480px) {
  .kpi-row { grid-template-columns: 1fr; }
  .trend-summary-strip { grid-template-columns: 1fr; }
}

/* ACCESSIBILITY: PREFERS REDUCED MOTION */
@media (prefers-reduced-motion: reduce) {
  .card-inner {
    transition: opacity 0.25s ease !important;
    transform-style: flat !important;
  }
  .card-wrapper.flipped .card-front {
    display: none !important;
    opacity: 0 !important;
  }
  .card-wrapper.flipped .card-back {
    transform: none !important;
    display: flex !important;
    opacity: 1 !important;
  }
  .card-wrapper:hover {
    transform: none !important;
  }
  .btn-flip {
    transition: none !important;
  }
  .card-highlight .card-face {
    animation: none !important;
  }
}

/* ================= 4. FILTER BANNER: DROPDOWNS & STICKY STYLES ================= */
.filter-title-hint {
  font-size: 13px;
  font-weight: 500;
  color: rgba(255, 255, 255, 0.92);
  margin-left: 6px;
}

.filter-dropdowns-grid {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 14px;
  width: 100%;
}

.filter-field-group {
  display: flex;
  flex-direction: column;
  gap: 5px;
}

.filter-field-label {
  display: flex;
  align-items: center;
  justify-content: space-between;
  font-size: 12px;
  font-weight: 800;
  color: #FFFFFF;
  text-transform: uppercase;
  letter-spacing: 0.5px;
}

.filter-active-dot {
  display: none;
  width: 7px;
  height: 7px;
  border-radius: 50%;
  background: #FFD166;
  box-shadow: 0 0 6px #FFD166;
}

.filter-field-select {
  height: 48px;
  min-height: 48px;
  border-radius: 12px;
  background: #FFFFFF;
  border: 2px solid transparent;
  color: var(--text-dark);
  font-size: 15px;
  font-weight: 600;
  padding: 0 14px;
  outline: none;
  cursor: pointer;
  box-shadow: 0 3px 8px rgba(0, 0, 0, 0.1);
  transition: all 0.2s cubic-bezier(0.4, 0, 0.2, 1);
  width: 100%;
}

.filter-field-select:hover {
  transform: translateY(-2px);
  box-shadow: 0 6px 16px rgba(0, 0, 0, 0.16);
}

.filter-field-select:focus-visible {
  outline: 2.5px solid #FFD166;
  outline-offset: 2px;
}

.filter-field-select.active-filter {
  border: 2.5px solid #FFD166 !important;
  background: #FFFDF9;
  font-weight: 700;
  color: #2B1A10;
}

/* STICKY BANNER COMPACT BEHAVIOR (~56px) */
.filter-banner-card.is-sticky {
  min-height: 56px;
  padding: 8px 18px !important;
  gap: 0 !important;
  box-shadow: 0 10px 30px rgba(120, 60, 10, 0.35);
}

.filter-banner-card.is-sticky .filter-banner-desc {
  display: none !important;
}

.filter-banner-card.is-sticky .btn-toggle-filter {
  display: inline-flex !important;
}

.filter-banner-card:not(.is-sticky) .btn-toggle-filter {
  display: none;
}

.filter-banner-card.is-sticky:not(.dropdowns-open) .filter-dropdowns-grid {
  display: none !important;
}

.filter-banner-card.is-sticky.dropdowns-open .filter-dropdowns-grid {
  display: grid !important;
  position: absolute;
  top: calc(100% + 6px);
  left: 0;
  right: 0;
  background: linear-gradient(135deg, #E8590C 0%, #F59F42 100%) !important;
  border-radius: 14px;
  box-shadow: 0 12px 32px rgba(0, 0, 0, 0.35);
  padding: 16px 20px;
  z-index: 650;
  border: 1px solid rgba(255, 255, 255, 0.25);
}

.btn-toggle-filter {
  display: none;
  background: #FFFFFF;
  color: var(--primary);
  border: 1.5px solid #FFFFFF;
  border-radius: 9999px;
  padding: 6px 14px;
  font-size: 13px;
  font-weight: 800;
  cursor: pointer;
  align-items: center;
  gap: 5px;
  transition: all 0.2s ease;
  box-shadow: 0 2px 6px rgba(0, 0, 0, 0.15);
}

.btn-toggle-filter:hover {
  background: var(--bg-cream);
  transform: translateY(-1px);
}

/* BQ-01 SUMMARY STRIP (4 COLUMNS) */
.trend-summary-strip {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 10px;
  margin-top: 12px;
  background: #FFF1E6;
  border: 1px solid #F5D3BC;
  border-radius: 12px;
  padding: 10px 14px;
}

.strip-col {
  display: flex;
  align-items: center;
  gap: 10px;
}

.strip-icon {
  font-size: 20px;
}

.strip-info {
  display: flex;
  flex-direction: column;
  gap: 2px;
}

.strip-lbl {
  font-size: 11px;
  font-weight: 700;
  color: var(--text-muted);
  text-transform: uppercase;
}

.strip-val {
  font-size: 13.5px;
  font-weight: 800;
  color: var(--text-dark);
}

/* BQ-03 DONUT & LEGEND LAYOUT */
.donut-layout {
  display: flex;
  align-items: center;
  justify-content: space-around;
  gap: 16px;
  flex-wrap: wrap;
  flex: 1;
  padding: 8px 0;
}

.donut-svg-wrap {
  position: relative;
  width: 150px;
  height: 150px;
  flex-shrink: 0;
  display: flex;
  align-items: center;
  justify-content: center;
}

.donut-center-info {
  position: absolute;
  top: 50%;
  left: 50%;
  transform: translate(-50%, -50%);
  text-align: center;
  pointer-events: none;
  display: flex;
  flex-direction: column;
  align-items: center;
  justify-content: center;
}

.donut-center-num {
  font-size: 24px;
  font-weight: 800;
  color: var(--text-dark);
  line-height: 1.1;
}

.donut-center-lbl {
  font-size: 11px;
  font-weight: 700;
  color: var(--text-muted);
  letter-spacing: 0.5px;
  text-transform: uppercase;
}

.donut-legend {
  display: flex;
  flex-direction: column;
  gap: 6px;
  flex: 1;
  min-width: 180px;
}

.donut-legend-item {
  display: flex;
  align-items: center;
  justify-content: space-between;
  font-size: 13.5px;
  font-weight: 600;
  padding: 2px 0;
}

.legend-left {
  display: flex;
  align-items: center;
  gap: 8px;
}

.legend-dot {
  width: 10px;
  height: 10px;
  border-radius: 50%;
  flex-shrink: 0;
}

.legend-name {
  font-weight: 600;
}

.legend-val {
  font-weight: 700;
}

/* BQ-04 BODY & HORIZONTAL RATING CHART */
.bq4-layout {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 14px;
}

.bq4-col {
  border-radius: 12px;
  padding: 12px 14px;
  display: flex;
  flex-direction: column;
  gap: 8px;
}

.bq4-col-late {
  background: #FEF2F2;
  border: 1px solid #FECACA;
}

.bq4-col-ontime {
  background: #F0FDF4;
  border: 1px solid #BBF7D0;
}

.bq4-col-header {
  display: flex;
  align-items: center;
  justify-content: space-between;
}

.bq4-col-badge {
  font-size: 12px;
  font-weight: 800;
  padding: 3px 8px;
  border-radius: 6px;
}

.badge-late {
  background: #FEE2E2;
  color: #DC2626;
}

.badge-ontime {
  background: #DCFCE7;
  color: #1B9E6B;
}

.bq4-col-n {
  font-size: 11.5px;
  font-weight: 600;
  color: var(--text-muted);
}

.bq4-stat-row {
  display: flex;
  gap: 16px;
}

.bq4-metric-box {
  display: flex;
  flex-direction: column;
  gap: 2px;
}

.bq4-metric-val {
  font-size: 32px;
  font-weight: 800;
  line-height: 1;
}

.val-late { color: #DC2626; }
.val-ontime { color: #1B9E6B; }

.bq4-metric-lbl {
  font-size: 11px;
  font-weight: 600;
  color: var(--text-muted);
}

.bq4-bar-container {
  display: flex;
  flex-direction: column;
  gap: 4px;
}

.bq4-bar-lbl {
  display: flex;
  justify-content: space-between;
  font-size: 11.5px;
  font-weight: 600;
  color: var(--text-muted);
}

.bq4-bar-track {
  height: 8px;
  background: rgba(0,0,0,0.06);
  border-radius: 9999px;
  overflow: hidden;
}

.bq4-bar-fill {
  height: 100%;
  border-radius: 9999px;
  transition: width 0.4s ease;
}

.fill-late { background: #DC2626; }
.fill-ontime { background: #1B9E6B; }

.bq4-chart-wrap {
  background: #FFFFFF;
  border: 1px solid #EADCD0;
  border-radius: 12px;
  padding: 10px 16px;
  display: flex;
  flex-direction: column;
  gap: 6px;
}

.bq4-chart-title {
  font-size: 12px;
  font-weight: 700;
  color: var(--text-muted);
}

.bq4-bar-row {
  display: flex;
  align-items: center;
  gap: 10px;
}

.bq4-bar-label {
  width: 85px;
  font-size: 12px;
  font-weight: 700;
  color: var(--text-dark);
}

.bq4-scale-track {
  flex: 1;
  height: 20px;
  background: #F3E5D8;
  border-radius: 6px;
  overflow: hidden;
  position: relative;
}

.bq4-scale-fill {
  height: 100%;
  border-radius: 6px;
  display: flex;
  align-items: center;
  justify-content: flex-end;
  padding-right: 8px;
  transition: width 0.5s ease;
}

.bq4-bar-late {
  background: linear-gradient(90deg, #F87171, #DC2626);
}

.bq4-bar-ontime {
  background: linear-gradient(90deg, #34D399, #1B9E6B);
}

.bq4-bar-val {
  font-size: 11.5px;
  font-weight: 800;
  color: #FFFFFF;
}

.bq4-axis {
  display: flex;
  justify-content: space-between;
  padding-left: 95px;
  font-size: 10px;
  font-weight: 700;
  color: var(--text-muted);
}

.bq4-note {
  font-size: 11.5px;
  font-weight: 600;
  color: var(--text-muted);
  font-style: italic;
  padding-left: 2px;
}
/* ================= BQ CARDS: BADGE, HEADLINE & INSIGHT FLIP PILL ================= */
.bq-badge {
  display: inline-block;
  background: var(--primary);
  color: #FFFFFF;
  font-size: 11px;
  font-weight: 800;
  padding: 2px 7px;
  border-radius: 6px;
  letter-spacing: 0.5px;
  text-transform: uppercase;
}

.card-title-group {
  display: flex;
  flex-direction: column;
  gap: 3px;
}

.card-title-row {
  display: flex;
  align-items: center;
  gap: 8px;
}

.card-headline {
  font-size: 13px;
  font-weight: 600;
  color: var(--primary-hover);
  margin-top: 2px;
  line-height: 1.35;
}

.card-header-actions {
  display: flex;
  align-items: center;
  gap: 8px;
}

.btn-flip-insight {
  display: inline-flex;
  align-items: center;
  gap: 5px;
  background: var(--primary-subtle);
  border: 1.5px solid rgba(232, 89, 12, 0.25);
  border-radius: 9999px;
  padding: 5px 13px;
  font-size: 12.5px;
  font-weight: 700;
  color: var(--primary);
  cursor: pointer;
  transition: all 0.2s ease;
}

.btn-flip-insight:hover {
  background: var(--primary);
  color: #FFFFFF;
  border-color: var(--primary);
  transform: translateY(-1px);
  box-shadow: 0 3px 8px rgba(232, 89, 12, 0.25);
}

.btn-flip-insight:active {
  transform: scale(0.96);
}

.btn-flip-insight:focus-visible {
  outline: 2px solid var(--primary);
  outline-offset: 2px;
}

/* BQ BACK FACE STRUCTURE (EXACT SAME HEIGHT, 3 SECTIONS) */
.bq-card-back {
  display: flex !important;
  flex-direction: column;
  padding: 16px 18px !important;
  height: 100% !important;
  box-sizing: border-box !important;
  overflow-y: auto !important;
  gap: 10px;
}

.bq-back-top {
  display: flex;
  align-items: center;
  justify-content: space-between;
  border-bottom: 1.5px solid var(--border-light);
  padding-bottom: 8px;
}

.bq-back-title {
  display: flex;
  align-items: center;
  gap: 8px;
}

.bq-back-title h4 {
  font-size: 15px;
  font-weight: 800;
  color: var(--text-dark);
  margin: 0;
}

.btn-flip-close {
  background: var(--bg-cream);
  border: 1px solid var(--border-light);
  width: 28px;
  height: 28px;
  border-radius: 50%;
  display: flex;
  align-items: center;
  justify-content: center;
  cursor: pointer;
  font-size: 13px;
  font-weight: 700;
  color: var(--text-dark);
  transition: all 0.2s ease;
}

.btn-flip-close:hover {
  background: var(--primary);
  color: #FFFFFF;
}

.bq-back-body {
  display: flex;
  flex-direction: column;
  gap: 8px;
  flex: 1;
}

.bq-section-box {
  border-radius: 9px;
  padding: 9px 12px;
  display: flex;
  flex-direction: column;
  gap: 3px;
}

.bq-box-meaning {
  background: var(--bg-cream);
  border: 1px solid var(--border-light);
}

.bq-box-meaning .bq-box-title {
  color: var(--text-muted);
  font-size: 11px;
  font-weight: 800;
  text-transform: uppercase;
  letter-spacing: 0.4px;
}

.bq-box-finding {
  background: #FFF3C4 !important;
  border: 1px solid #FFE69C !important;
}

.bq-box-finding .bq-box-title {
  color: #7A5B04 !important;
  font-size: 11px;
  font-weight: 800;
  text-transform: uppercase;
  letter-spacing: 0.4px;
}

.bq-box-business {
  background: #FFE8D6 !important;
  border: 1px solid #F5D3BC !important;
}

.bq-box-business .bq-box-title {
  color: #C2410C !important;
  font-size: 11px;
  font-weight: 800;
  text-transform: uppercase;
  letter-spacing: 0.4px;
}

.bq-box-desc {
  font-size: 14px;
  line-height: 1.4;
  color: var(--text-dark);
  font-weight: 500;
}

/* ================= RINGKASAN TEMUAN (F-01 s/d F-05) ================= */
.findings-list {
  display: flex;
  flex-direction: column;
  gap: 7px;
  margin-top: 4px;
}

.finding-item {
  display: flex;
  align-items: center;
  gap: 10px;
  background: var(--bg-cream);
  border: 1px solid var(--border-light);
  border-radius: 8px;
  padding: 8px 12px;
  cursor: pointer;
  transition: all 0.2s ease;
  font-size: 12.5px;
  color: var(--text-dark);
  user-select: none;
}

.finding-item:hover {
  background: var(--primary-subtle);
  border-color: rgba(232, 89, 12, 0.35);
  transform: translateX(3px);
  box-shadow: 0 2px 6px rgba(232, 89, 12, 0.12);
}

.finding-badge {
  background: var(--primary);
  color: #FFFFFF;
  font-size: 11px;
  font-weight: 800;
  padding: 2px 6px;
  border-radius: 5px;
  letter-spacing: 0.4px;
  flex-shrink: 0;
}

.finding-text {
  flex: 1;
  line-height: 1.35;
  font-weight: 500;
}

.finding-arrow {
  color: var(--primary);
  font-weight: 700;
  font-size: 14px;
  opacity: 0.6;
  transition: transform 0.2s ease, opacity 0.2s ease;
}

.finding-item:hover .finding-arrow {
  opacity: 1;
  transform: translateX(3px);
}

</style>
</head>
<body>

<div class="app-layout">

  <!-- SIDEBAR KIRI (200px, Sticky setinggi layar) -->
  <aside class="sidebar">
    <div class="sidebar-top">
      <div class="sidebar-brand">
        <div class="brand-title">🍲 SwiftBite</div>
        <div class="brand-subtitle">PERFORMANCE ANALYTICS</div>
      </div>
      
      <ul class="sidebar-menu">
        <li>
          <button class="menu-btn active" id="menu-overview" onclick="navigateMenu('overview')">
            <span class="menu-icon">📊</span>
            <span>Overview</span>
          </button>
        </li>
        <li>
          <button class="menu-btn" id="menu-transaksi" onclick="navigateMenu('transaksi')">
            <span class="menu-icon">💳</span>
            <span>Transaksi (BQ-01)</span>
          </button>
        </li>
        <li>
          <button class="menu-btn" id="menu-pengiriman" onclick="navigateMenu('pengiriman')">
            <span class="menu-icon">🚚</span>
            <span>Pengiriman (BQ-02)</span>
          </button>
        </li>
        <li>
          <button class="menu-btn" id="menu-review" onclick="navigateMenu('review')">
            <span class="menu-icon">⭐</span>
            <span>Review & Sentimen (BQ-03)</span>
          </button>
        </li>
        <li>
          <button class="menu-btn" id="menu-keterlambatan" onclick="navigateMenu('keterlambatan')">
            <span class="menu-icon">⏱️</span>
            <span>Keterlambatan & Rating (BQ-04)</span>
          </button>
        </li>
        <li>
          <button class="menu-btn" id="menu-pembatalan" onclick="navigateMenu('pembatalan')">
            <span class="menu-icon">🛑</span>
            <span>Pembatalan (BQ-05)</span>
          </button>
        </li>
        <li>
          <button class="menu-btn" id="menu-insights" onclick="navigateMenu('insights')">
            <span class="menu-icon">💡</span>
            <span>Insights</span>
          </button>
        </li>
      </ul>
    </div>

    <!-- Ruang Bawah Menu: Ringkasan Cepat + Mode Penjelasan + Variant -->
    <div class="sidebar-bottom-group">
      <div class="sidebar-summary-card">
        <div class="sidebar-summary-title">⚡ Ringkasan Cepat</div>
        <div class="sidebar-summary-item">
          <span class="ss-lbl">Total Pesanan</span>
          <span class="ss-val" id="sbOrders">335</span>
        </div>
        <div class="sidebar-summary-item">
          <span class="ss-lbl">Keterlambatan</span>
          <span class="ss-val text-danger" id="sbLate">31,2%</span>
        </div>
        <div class="sidebar-summary-item">
          <span class="ss-lbl">Review Coverage</span>
          <span class="ss-val text-success" id="sbCov">57,3%</span>
        </div>
      </div>

      <div class="sidebar-bottom">
        <button class="btn-sidebar-explain" id="btnExplainAll" onclick="toggleAllExplanation()">
          <span>📖</span> Mode penjelasan
        </button>
        <div class="sidebar-variant-tag">Variant __VARIANT__</div>
      </div>
    </div>
  </aside>

  <!-- MAIN AREA (PENUH LEBAR) -->
  <main class="main-wrapper">

    <!-- GREETING & PAGE TITLE -->
    <header class="greeting-section">
      <div class="page-title">SwiftBite Insight Hub</div>
      <div class="page-subtitle">Pesanan, Pengiriman, dan Kepuasan Pelanggan · Januari–Maret 2026</div>
      <div class="greeting-headline" id="greetingTitle">Selamat datang, Ghina! 👋</div>
      <div class="greeting-sub">Ini ringkasan performa SwiftBite, Januari–Maret 2026 (Variant __VARIANT__).</div>
    </header>

    <!-- 1. BANNER FILTER PALING MENONJOL (GRADASI ORANYE #E8590C ke #F59F42) -->
    <div class="filter-banner-card" id="filterBanner">
      <div class="filter-banner-top">
        <div class="filter-banner-titlebox">
          <div class="filter-banner-title">
            <svg class="funnel-icon" width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" style="display:inline-block; vertical-align:-3px; margin-right:4px;"><polygon points="22 3 2 3 10 12.46 10 19 14 21 14 12.46 22 3"></polygon></svg>
            <span>Filter data</span>
          </div>
          <div class="filter-banner-desc">Pilih bulan, zona, atau sentimen untuk mengubah semua angka</div>
        </div>

        <!-- Chip Filter Aktif dengan tombol ✕ -->
        <div class="filter-chips-bar" id="filterChipsBar">
          <span class="filter-chip-none">Semua data aktif (tidak ada filter terpilih)</span>
        </div>

        <div class="filter-banner-top-right">
          <!-- Penghitung hidup di kanan banner -->
          <div class="filter-live-counter" id="filterCounterBox">
            Menampilkan <span class="counter-num" id="cntFiltered">335</span> dari <span id="cntTotal">335</span> pesanan
          </div>

          <!-- Tombol Ubah Filter saat Sticky -->
          <button id="btnToggleFilterPanel" class="btn-toggle-filter" aria-expanded="false" title="Buka / tutup panel filter">
            Ubah filter ▾
          </button>

          <!-- Tombol Reset Putih Berkontras Tinggi -->
          <button id="reset" class="btn-reset-white" disabled title="Kembalikan semua filter ke awal">
            <span>🔄</span> Reset
          </button>
        </div>
      </div>

      <!-- Tiga Dropdown (Bulan, Zona, Sentimen) Tinggi 48px -->
      <div class="filter-dropdowns-grid" id="filterDropdownsGrid">
        <div class="filter-field-group" id="grpMonth">
          <label for="fMonth" class="filter-field-label">
            <span>📅 Bulan</span>
            <span class="filter-active-dot" id="dotMonth"></span>
          </label>
          <select id="fMonth" class="filter-field-select" aria-label="Pilih Bulan"></select>
        </div>

        <div class="filter-field-group" id="grpZone">
          <label for="fZone" class="filter-field-label">
            <span>📍 Zona</span>
            <span class="filter-active-dot" id="dotZone"></span>
          </label>
          <select id="fZone" class="filter-field-select" aria-label="Pilih Zona"></select>
        </div>

        <div class="filter-field-group" id="grpSent">
          <label for="fSent" class="filter-field-label">
            <span>💬 Sentimen</span>
            <span class="filter-active-dot" id="dotSent"></span>
          </label>
          <select id="fSent" class="filter-field-select" aria-label="Pilih Sentimen"></select>
        </div>
      </div>
    </div>

    <section class="kpi-row" id="sectionKPIs">

      <!-- KPI 1: Total Orders -->
      <div class="card-wrapper kpi-card-wrap" id="card-kpi-1">
        <div class="card-inner">
          <div class="card-face card-front kpi-card-front">
            <button class="btn-flip" aria-label="Penjelasan Total Orders" aria-pressed="false" onclick="toggleCard('card-kpi-1')">i</button>
            <div class="kpi-top">
              <div class="kpi-icon-box">🛍️</div>
              <span class="kpi-title">Total Orders</span>
            </div>
            <div class="kpi-middle">
              <div class="kpi-value" id="k1">335</div>
              <svg class="kpi-sparkline" id="sp1" viewBox="0 0 60 26"></svg>
            </div>
            <div class="kpi-bottom" id="k1s">
              <span class="trend-badge trend-up">↑ +24,0% vs Feb</span>
            </div>
          </div>
          <div class="card-face card-back">
            <button class="btn-flip" aria-label="Tutup penjelasan" aria-pressed="true" onclick="toggleCard('card-kpi-1')">✕</button>
            <div class="back-header"><h4>Total Orders</h4></div>
            <div class="back-meaning">Jumlah pesanan unik pada periode dan filter yang dipilih.</div>
            <div class="back-kpi-sub" id="bk1-f">335 pesanan</div>
          </div>
        </div>
      </div>

      <!-- KPI 2: Net Payment -->
      <div class="card-wrapper kpi-card-wrap" id="card-kpi-2">
        <div class="card-inner">
          <div class="card-face card-front kpi-card-front">
            <button class="btn-flip" aria-label="Penjelasan Net Payment" aria-pressed="false" onclick="toggleCard('card-kpi-2')">i</button>
            <div class="kpi-top">
              <div class="kpi-icon-box">💳</div>
              <span class="kpi-title">Net Payment</span>
            </div>
            <div class="kpi-middle">
              <div class="kpi-value" id="k2">Rp37.161.000</div>
              <svg class="kpi-sparkline" id="sp2" viewBox="0 0 60 26"></svg>
            </div>
            <div class="kpi-bottom" id="k2s">
              <span class="trend-badge trend-up">↑ +18,7% vs Feb</span>
            </div>
          </div>
          <div class="card-face card-back">
            <button class="btn-flip" aria-label="Tutup penjelasan" aria-pressed="true" onclick="toggleCard('card-kpi-2')">✕</button>
            <div class="back-header"><h4>Net Payment</h4></div>
            <div class="back-meaning">Uang yang benar-benar diterima: pembayaran sukses dikurangi refund.</div>
            <div class="back-kpi-sub" id="bk2-f">Rp37.161.000</div>
          </div>
        </div>
      </div>

      <!-- KPI 3: Review Coverage -->
      <div class="card-wrapper kpi-card-wrap" id="card-kpi-3">
        <div class="card-inner">
          <div class="card-face card-front kpi-card-front">
            <button class="btn-flip" aria-label="Penjelasan Review Coverage" aria-pressed="false" onclick="toggleCard('card-kpi-3')">i</button>
            <div class="kpi-top">
              <div class="kpi-icon-box">📝</div>
              <span class="kpi-title">Review Coverage</span>
            </div>
            <div class="kpi-middle">
              <div class="kpi-value" id="k3">57,3%</div>
              <svg class="kpi-sparkline" id="sp3" viewBox="0 0 60 26"></svg>
            </div>
            <div class="kpi-bottom" id="k3s">192 / 335 order</div>
          </div>
          <div class="card-face card-back">
            <button class="btn-flip" aria-label="Tutup penjelasan" aria-pressed="true" onclick="toggleCard('card-kpi-3')">✕</button>
            <div class="back-header"><h4>Review Coverage</h4></div>
            <div class="back-meaning">Berapa persen pesanan yang diberi ulasan. Pesanan tanpa ulasan tetap dihitung.</div>
            <div class="back-kpi-sub" id="bk3-f">192 dari 335 pesanan</div>
          </div>
        </div>
      </div>

      <!-- KPI 4: Late Delivery Rate -->
      <div class="card-wrapper kpi-card-wrap" id="card-kpi-4">
        <div class="card-inner">
          <div class="card-face card-front kpi-card-front">
            <button class="btn-flip" aria-label="Penjelasan Late Delivery Rate" aria-pressed="false" onclick="toggleCard('card-kpi-4')">i</button>
            <div class="kpi-top">
              <div class="kpi-icon-box">⏱️</div>
              <span class="kpi-title">Late Delivery Rate</span>
            </div>
            <div class="kpi-middle">
              <div class="kpi-value" id="k4">31,2%</div>
              <svg class="kpi-sparkline" id="sp4" viewBox="0 0 60 26"></svg>
            </div>
            <div class="kpi-bottom" id="k4s">92 / 295 valid</div>
          </div>
          <div class="card-face card-back">
            <button class="btn-flip" aria-label="Tutup penjelasan" aria-pressed="true" onclick="toggleCard('card-kpi-4')">✕</button>
            <div class="back-header"><h4>Late Delivery Rate</h4></div>
            <div class="back-meaning">Persentase pesanan terkirim yang terlambat. Makin kecil makin baik.</div>
            <div class="back-kpi-sub" id="bk4-f">92 dari 295 pesanan terkirim</div>
          </div>
        </div>
      </div>

      <!-- KPI 5: Rata-rata Rating -->
      <div class="card-wrapper kpi-card-wrap" id="card-kpi-5">
        <div class="card-inner">
          <div class="card-face card-front kpi-card-front">
            <button class="btn-flip" aria-label="Penjelasan Rata-rata Rating" aria-pressed="false" onclick="toggleCard('card-kpi-5')">i</button>
            <div class="kpi-top">
              <div class="kpi-icon-box">⭐</div>
              <span class="kpi-title">Rata-rata Rating</span>
            </div>
            <div class="kpi-middle">
              <div class="kpi-value" id="k5">3,58</div>
              <svg class="kpi-sparkline" id="sp5" viewBox="0 0 60 26"></svg>
            </div>
            <div class="kpi-bottom" id="k5s">n = 192 review</div>
          </div>
          <div class="card-face card-back">
            <button class="btn-flip" aria-label="Tutup penjelasan" aria-pressed="true" onclick="toggleCard('card-kpi-5')">✕</button>
            <div class="back-header"><h4>Rata-rata Rating</h4></div>
            <div class="back-meaning">Rata-rata bintang dari pesanan yang diulas saja.</div>
            <div class="back-kpi-sub" id="bk5-f">Total skor dari 192 ulasan</div>
          </div>
        </div>
      </div>

    </section>

    <!-- 3. BARIS TENGAH (2fr : 1fr) -->
    <section class="middle-row" id="sectionMiddle">

      <!-- B. GRAFIK TREN BULANAN: DUA PANEL BERDAMPINGAN DENGAN STRIP TEMUAN & MAKNA BISNIS -->
      <div class="card-wrapper" id="card-chart-trend">
        <div class="card-inner">
          <div class="card-face card-front chart-card-front">
            <button class="btn-flip" aria-label="Penjelasan Tren Bulanan" aria-pressed="false" onclick="toggleCard('card-chart-trend')">i</button>
            <div class="card-header-bar">
              <h3 class="card-heading">Tren Bulanan</h3>
              <div class="tabs-group">
                <button class="tab-btn active" id="tab-combo" onclick="setTrendTab('combo')">Net Payment & Orders</button>
                <button class="tab-btn" id="tab-cancel" onclick="setTrendTab('cancel')">Cancel rate</button>
              </div>
            </div>

            <!-- DUA PANEL BERDAMPINGAN -->
            <div class="trend-panels-grid">
              <!-- Panel Kiri -->
              <div class="trend-subpanel" id="panelLeft">
                <div class="panel-header">
                  <span class="panel-title" id="panelLeftTitle">Net Payment (Rp juta)</span>
                  <div class="panel-kpi-box">
                    <span class="panel-big-num" id="panelLeftTotal">Rp37,16 jt</span>
                    <span class="panel-sub-lbl" id="panelLeftSub">Total periode</span>
                  </div>
                </div>
                <div class="panel-chart-wrap" id="panelLeftChart"></div>
              </div>

              <!-- Panel Kanan -->
              <div class="trend-subpanel" id="panelRight">
                <div class="panel-header">
                  <span class="panel-title" id="panelRightTitle">Jumlah pesanan</span>
                  <div class="panel-kpi-box">
                    <span class="panel-big-num" id="panelRightTotal">335</span>
                    <span class="panel-sub-lbl" id="panelRightSub">Total pesanan</span>
                  </div>
                </div>
                <div class="panel-chart-wrap" id="panelRightChart"></div>
              </div>
            </div>

            <!-- Strip Ringkasan 4 Kolom -->
            <div class="trend-summary-strip" id="trendSummaryStrip">
              <div class="strip-col">
                <span class="strip-icon">💰</span>
                <div class="strip-info">
                  <span class="strip-lbl">Net Payment Tertinggi</span>
                  <span class="strip-val" id="stPeakNet">Mar: Rp14,09 jt</span>
                </div>
              </div>
              <div class="strip-col">
                <span class="strip-icon">🛍️</span>
                <div class="strip-info">
                  <span class="strip-lbl">Pesanan Tertinggi</span>
                  <span class="strip-val" id="stPeakOrd">Mar: 129 pesanan</span>
                </div>
              </div>
              <div class="strip-col">
                <span class="strip-icon">🛑</span>
                <div class="strip-info">
                  <span class="strip-lbl">Batal Tertinggi</span>
                  <span class="strip-val" id="stPeakCanc">Mar: 14,7%</span>
                </div>
              </div>
              <div class="strip-col">
                <span class="strip-icon">📊</span>
                <div class="strip-info">
                  <span class="strip-lbl">Total Periode</span>
                  <span class="strip-val" id="stTotalPeriod">Rp37.161.000 (335 ord)</span>
                </div>
              </div>
            </div>

            <!-- 3. STRIP TEMUAN UTAMA & MAKNA BISNIS (TREN BULANAN) -->
            <div class="card-insight-strip">
              <div class="strip-box strip-box-findings">
                <div class="strip-header">
                  <span class="strip-header-icon">🔍</span>
                  <span class="strip-header-title">Temuan Utama</span>
                </div>
                <div class="strip-text" id="stripTrendFinding">Memuat analisis...</div>
              </div>
              <div class="strip-box strip-box-business">
                <div class="strip-header">
                  <span class="strip-header-icon">💡</span>
                  <span class="strip-header-title">Makna Bisnis</span>
                </div>
                <div class="strip-text" id="stripTrendBusiness">Memuat makna bisnis...</div>
              </div>
            </div>

          </div>

          <div class="card-face card-back">
            <button class="btn-flip" aria-label="Tutup penjelasan" aria-pressed="true" onclick="toggleCard('card-chart-trend')">✕</button>
            <div class="back-header"><h4>Tren Bulanan</h4></div>
            <div class="back-meaning">Naik-turun Net Payment dan jumlah pesanan tiap bulan.</div>
          </div>
        </div>
      </div>

      <!-- KANAN: Donut Sentimen Review DENGAN STRIP TEMUAN & MAKNA BISNIS -->
      <div class="card-wrapper" id="card-chart-sent">
        <div class="card-inner">
          <div class="card-face card-front chart-card-front">
            <button class="btn-flip" aria-label="Penjelasan Sentimen Review" aria-pressed="false" onclick="toggleCard('card-chart-sent')">i</button>
            <div class="card-header-bar">
              <h3 class="card-heading">Sentimen Review</h3>
            </div>
            <div class="donut-layout">
              <div class="donut-svg-wrap">
                <svg viewBox="0 0 160 160" id="svgDonut" width="150" height="150"></svg>
                <div class="donut-center-info">
                  <div class="donut-center-num" id="donutTotalText">192</div>
                  <div class="donut-center-lbl">REVIEW</div>
                </div>
              </div>
              <div class="donut-legend" id="donutLegend"></div>
            </div>

          </div>

          <div class="card-face card-back bq-card-back">
            <div class="bq-back-top">
              <div class="bq-back-title">
                <span class="bq-badge">BQ-05</span>
                <h4>Status Pesanan</h4>
              </div>
              <button class="btn-flip-close" aria-label="Tutup penjelasan" onclick="toggleCard('card-status')">✕</button>
            </div>
            <div class="bq-back-body">
              <div class="bq-section-box bq-box-meaning">
                <div class="bq-box-title">Arti grafik</div>
                <div class="bq-box-desc">Berapa pesanan selesai, gagal bayar, dan dikembalikan.</div>
              </div>
              <div class="bq-section-box bq-box-finding">
                <div class="bq-box-title">🔍 Temuan utama</div>
                <div class="bq-box-desc" id="stripStatusFinding">Memuat analisis...</div>
              </div>
              <div class="bq-section-box bq-box-business">
                <div class="bq-box-title">💡 Makna bisnis</div>
                <div class="bq-box-desc" id="stripStatusBusiness">Memuat makna bisnis...</div>
              </div>
            </div>
          </div>
        </div>
      </div>

      <!-- KARTU 3: Business Insights -->
      <div class="card-wrapper" id="card-insights">
        <div class="card-inner">
          <div class="card-face card-front bottom-card-front">
            <button class="btn-flip" aria-label="Penjelasan Ringkasan Temuan" aria-pressed="false" onclick="toggleCard('card-insights')">i</button>
            <div class="card-header-bar">
              <h3 class="card-heading">Ringkasan Temuan</h3>
            </div>
            <div class="findings-list" id="insightsList"></div>
          </div>
          <div class="card-face card-back">
            <button class="btn-flip" aria-label="Tutup penjelasan" aria-pressed="true" onclick="toggleCard('card-insights')">✕</button>
            <div class="back-header"><h4>Ringkasan Temuan</h4></div>
            <div class="back-meaning">Rangkuman 5 temuan utama (F-01 s/d F-05) yang terhubung ke menu analisis masing-masing.</div>
          </div>
        </div>
      </div>

    </section>

  </main>
</div>

<script>
// ================= DATA & INITIALIZATION =================
const DATA = __DATA__;
const $ = id => document.getElementById(id);
const setTxt = (id, val) => { const el = $(id); if (el) el.textContent = val; };
const uniq = k => [...new Set(DATA.map(d => d[k]).filter(Boolean))].sort();
const SENT_ORDER = ["Positive", "Neutral", "Negative", "No Text", "Unknown"];
const TOTAL_ALL_ORDERS = DATA.length;

// 1. FILTER STATE (Single source of truth)
const state = {
  month: "ALL",
  zone: "ALL",
  sentiment: "ALL"
};

// filtered() strictly reads from state, never directly touches DOM
function filtered() {
  return DATA.filter(d =>
    (state.month === "ALL" || d.m === state.month) &&
    (state.zone === "ALL" || d.z === state.zone) &&
    (state.sentiment === "ALL" || d.s === state.sentiment)
  );
}

let activeTrendTab = "combo"; // "combo" or "cancel"
let activeNavMenu = "overview";
let explainModeAll = false;

// 1. Time-Based Greeting (Selamat ..., Ghina! 👋)
function updateGreeting() {
  const h = new Date().getHours();
  let greet = "Selamat malam, Ghina! 👋";
  if (h >= 5 && h < 11) greet = "Selamat pagi, Ghina! 👋";
  else if (h >= 11 && h < 15) greet = "Selamat siang, Ghina! 👋";
  else if (h >= 15 && h < 18) greet = "Selamat sore, Ghina! 👋";
  setTxt("greetingTitle", greet);
}
updateGreeting();

// 2. Sidebar Navigation (7 Functional Menus)
function navigateMenu(menuId) {
  activeNavMenu = menuId;
  document.querySelectorAll(".menu-btn").forEach(btn => btn.classList.remove("active"));
  const btn = $("menu-" + menuId);
  if (btn) btn.classList.add("active");

  const kpis = $("sectionKPIs");
  const middle = $("sectionMiddle");
  const bq04 = $("sectionBQ04");
  const bottom = $("sectionBottom");
  const cTrend = $("card-chart-trend");
  const cSent = $("card-chart-sent");
  const cZones = $("card-zones");
  const cStatus = $("card-status");
  const cInsights = $("card-insights");

  if (!kpis || !middle || !bq04 || !bottom) return;

  kpis.style.display = "grid";
  middle.style.display = "grid";
  bq04.style.display = "grid";
  bottom.style.display = "grid";
  [cTrend, cSent, cZones, cStatus, cInsights].forEach(c => {
    if (c) {
      c.style.display = "block";
      c.classList.remove("section-full-width");
    }
  });

  if (menuId === "overview") {
    middle.style.gridTemplateColumns = "2fr 1fr";
    bottom.style.gridTemplateColumns = "repeat(3, 1fr)";
  } else if (menuId === "transaksi") {
    if (cSent) cSent.style.display = "none";
    bq04.style.display = "none";
    bottom.style.display = "none";
    middle.style.gridTemplateColumns = "1fr";
    if (cTrend) cTrend.classList.add("section-full-width");
  } else if (menuId === "pengiriman") {
    middle.style.display = "none";
    bq04.style.display = "none";
    if (cStatus) cStatus.style.display = "none";
    if (cInsights) cInsights.style.display = "none";
    bottom.style.gridTemplateColumns = "1fr";
    if (cZones) cZones.classList.add("section-full-width");
  } else if (menuId === "review") {
    if (cTrend) cTrend.style.display = "none";
    bq04.style.display = "none";
    bottom.style.display = "none";
    middle.style.gridTemplateColumns = "1fr";
    if (cSent) cSent.classList.add("section-full-width");
  } else if (menuId === "keterlambatan") {
    middle.style.display = "none";
    bottom.style.display = "none";
    bq04.style.display = "grid";
  } else if (menuId === "pembatalan") {
    middle.style.display = "none";
    bq04.style.display = "none";
    if (cZones) cZones.style.display = "none";
    if (cInsights) cInsights.style.display = "none";
    bottom.style.gridTemplateColumns = "1fr";
    if (cStatus) cStatus.classList.add("section-full-width");
  } else if (menuId === "insights") {
    middle.style.display = "none";
    bq04.style.display = "none";
    if (cZones) cZones.style.display = "none";
    if (cStatus) cStatus.style.display = "none";
    bottom.style.gridTemplateColumns = "1fr";
    if (cInsights) cInsights.classList.add("section-full-width");
  }

  setTimeout(() => render(), 40);
}

// 3. Card Flip & Mode Penjelasan
function toggleCard(cardId) {
  const card = $(cardId);
  if (!card) return;
  card.classList.toggle("flipped");
  const isFlipped = card.classList.contains("flipped");
  const btn = card.querySelector(".btn-flip") || card.querySelector(".btn-flip-insight");
  if (btn) {
    btn.setAttribute("aria-pressed", isFlipped ? "true" : "false");
  }
}

function toggleAllExplanation() {
  explainModeAll = !explainModeAll;
  const btnExpl = $("btnExplainAll");
  if (btnExpl) btnExpl.classList.toggle("active", explainModeAll);
  document.querySelectorAll(".card-wrapper").forEach(card => {
    card.classList.toggle("flipped", explainModeAll);
    const btn = card.querySelector(".btn-flip") || card.querySelector(".btn-flip-insight");
    if (btn) btn.setAttribute("aria-pressed", explainModeAll ? "true" : "false");
  });
}

// Esc to reset flips
window.addEventListener("keydown", e => {
  if (e.key === "Escape") {
    document.querySelectorAll(".card-wrapper.flipped").forEach(card => {
      card.classList.remove("flipped");
      const btn = card.querySelector(".btn-flip") || card.querySelector(".btn-flip-insight");
      if (btn) btn.setAttribute("aria-pressed", "false");
    });
    explainModeAll = false;
    const btnExpl = $("btnExplainAll");
    if (btnExpl) btnExpl.classList.remove("active");
  }
});

// 4. DROPDOWNS SETUP & SYNC
function fillDropdown(id, vals, allLabel, formatter) {
  const el = $(id);
  if (!el) return;
  el.innerHTML = "";
  const allOpt = document.createElement("option");
  allOpt.value = "ALL";
  allOpt.textContent = allLabel;
  el.appendChild(allOpt);

  vals.forEach(v => {
    const opt = document.createElement("option");
    opt.value = v;
    opt.textContent = formatter ? formatter(v) : v;
    el.appendChild(opt);
  });
}

const MONTH_LABELS = { "2026-01": "Januari 2026", "2026-02": "Februari 2026", "2026-03": "Maret 2026" };
const MONTH_SHORT = { "2026-01": "Jan 2026", "2026-02": "Feb 2026", "2026-03": "Mar 2026" };

fillDropdown("fMonth", uniq("m"), "Semua bulan", v => MONTH_LABELS[v] || v);
fillDropdown("fZone",  uniq("z"), "Semua zona");
fillDropdown("fSent",  SENT_ORDER, "Semua sentimen");

function syncDropdowns() {
  const elM = $("fMonth"); if (elM) elM.value = state.month;
  const elZ = $("fZone");  if (elZ) elZ.value = state.zone;
  const elS = $("fSent");  if (elS) elS.value = state.sentiment;

  const isMActive = state.month !== "ALL";
  const isZActive = state.zone !== "ALL";
  const isSActive = state.sentiment !== "ALL";

  if (elM) elM.classList.toggle("active-filter", isMActive);
  if (elZ) elZ.classList.toggle("active-filter", isZActive);
  if (elS) elS.classList.toggle("active-filter", isSActive);

  const dotM = $("dotMonth"); if (dotM) dotM.style.display = isMActive ? "inline-block" : "none";
  const dotZ = $("dotZone");  if (dotZ) dotZ.style.display = isZActive ? "inline-block" : "none";
  const dotS = $("dotSent");  if (dotS) dotS.style.display = isSActive ? "inline-block" : "none";

  const grpM = $("grpMonth"); if (grpM) grpM.classList.toggle("is-active", isMActive);
  const grpZ = $("grpZone");  if (grpZ) grpZ.classList.toggle("is-active", isZActive);
  const grpS = $("grpSent");  if (grpS) grpS.classList.toggle("is-active", isSActive);

  const hasFilter = isMActive || isZActive || isSActive;
  const btnReset = $("reset");
  if (btnReset) btnReset.disabled = !hasFilter;

  // Active Filter Chips
  const chipsBar = $("filterChipsBar");
  if (chipsBar) {
    if (!hasFilter) {
      chipsBar.innerHTML = `<span class="filter-chip-none">Semua data aktif (tidak ada filter terpilih)</span>`;
    } else {
      let chipsHtml = "";
      if (isMActive) {
        const mLabel = MONTH_SHORT[state.month] || state.month;
        chipsHtml += `<span class="filter-chip-item">${mLabel} <button class="filter-chip-remove" onclick="clearFilter('m')" title="Hapus filter bulan">✕</button></span>`;
      }
      if (isZActive) {
        chipsHtml += `<span class="filter-chip-item">Zona ${state.zone} <button class="filter-chip-remove" onclick="clearFilter('z')" title="Hapus filter zona">✕</button></span>`;
      }
      if (isSActive) {
        chipsHtml += `<span class="filter-chip-item">Sentimen ${state.sentiment} <button class="filter-chip-remove" onclick="clearFilter('s')" title="Hapus filter sentimen">✕</button></span>`;
      }
      chipsBar.innerHTML = chipsHtml;
    }
  }
}

// Clear individual filter
function clearFilter(type) {
  if (type === 'm') state.month = "ALL";
  if (type === 'z') state.zone = "ALL";
  if (type === 's') state.sentiment = "ALL";
  syncDropdowns();
  render(true);
}

// Listeners for dropdowns
["fMonth", "fZone", "fSent"].forEach(id => {
  const el = $(id);
  if (el) {
    el.addEventListener("change", () => {
      const elM = $("fMonth"); if (elM) state.month = elM.value;
      const elZ = $("fZone");  if (elZ) state.zone = elZ.value;
      const elS = $("fSent");  if (elS) state.sentiment = elS.value;
      syncDropdowns();
      render(true);
    });
  }
});

// Reset button listener
const btnReset = $("reset");
if (btnReset) {
  btnReset.addEventListener("click", () => {
    state.month = "ALL";
    state.zone = "ALL";
    state.sentiment = "ALL";
    syncDropdowns();
    render(true);
  });
}

// Sticky Toggle Filter Panel Listener
const btnTogglePanel = $("btnToggleFilterPanel");
const bannerEl = $("filterBanner");
if (btnTogglePanel && bannerEl) {
  btnTogglePanel.addEventListener("click", e => {
    e.stopPropagation();
    bannerEl.classList.toggle("dropdowns-open");
    const isOpen = bannerEl.classList.contains("dropdowns-open");
    btnTogglePanel.setAttribute("aria-expanded", isOpen ? "true" : "false");
  });
  document.addEventListener("click", e => {
    if (!bannerEl.contains(e.target) && bannerEl.classList.contains("dropdowns-open")) {
      bannerEl.classList.remove("dropdowns-open");
      btnTogglePanel.setAttribute("aria-expanded", "false");
    }
  });
}

// Sticky banner scroll listener
window.addEventListener("scroll", () => {
  if (!bannerEl) return;
  if (window.scrollY > 80) bannerEl.classList.add("is-sticky");
  else bannerEl.classList.remove("is-sticky");
});

// Formatting Utilities (Half-Even Rounding)
function rnd(v, d) {
  if (v === null || v === undefined || isNaN(v)) return "N/A";
  const m = Math.pow(10, d), x = v * m, f = Math.floor(x), df = x - f;
  let r;
  if (Math.abs(df - 0.5) < 1e-9) r = (f % 2 === 0) ? f : f + 1;
  else r = Math.round(x);
  return (r / m).toFixed(d);
}

function fmtMoney(v) {
  if (v === null || v === undefined || isNaN(v)) return "N/A";
  const r = Math.round(v);
  return "Rp" + r.toString().replace(/\B(?=(\d{3})+(?!\d))/g, ".");
}

// Trend Tabs Sync
function setTrendTab(tab) {
  activeTrendTab = tab;
  syncTrendTabs();
  render(false);
}

function syncTrendTabs() {
  const btnCombo = $("tab-combo");
  const btnCancel = $("tab-cancel");
  if (btnCombo) btnCombo.classList.toggle("active", activeTrendTab === "combo");
  if (btnCancel) btnCancel.classList.toggle("active", activeTrendTab === "cancel");
}

// Sparklines for KPI Cards
function drawSparkline(id, points, color) {
  const el = $(id);
  if (!el) return;
  const vals = points.filter(p => p !== null && !isNaN(p));
  if (vals.length < 2 || vals.every(v => v === 0)) {
    el.innerHTML = '<line x1="2" y1="13" x2="58" y2="13" stroke="#D1C2B5" stroke-dasharray="2,2" stroke-width="1.5"/>';
    return;
  }
  const min = Math.min(...vals), max = Math.max(...vals);
  const range = (max - min) || 1;
  const w = 60, h = 26, pad = 3;
  const pts = vals.map((v, i) => {
    const x = pad + (i / (vals.length - 1)) * (w - 2 * pad);
    const y = (h - pad) - ((v - min) / range) * (h - 2 * pad);
    return `${x.toFixed(1)},${y.toFixed(1)}`;
  });
  el.innerHTML = `<polyline fill="none" stroke="${color}" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" points="${pts.join(' ')}"/>`;
}

// Clean Nice Scale function: 4 equal intervals (5 ticks)
function getNiceScale(maxVal, isCurrency) {
  if (isCurrency) {
    if (maxVal <= 0) return { top: 16, ticks: [0, 4, 8, 12, 16] };
    let top = 4;
    while (top < maxVal) {
      top += 4;
    }
    const step = top / 4;
    return { top, ticks: [0, step, step * 2, step * 3, top] };
  } else {
    if (maxVal <= 0) return { top: 160, ticks: [0, 40, 80, 120, 160] };
    let top = 40;
    while (top < maxVal) {
      top += 40;
    }
    const step = top / 4;
    return { top, ticks: [0, step, step * 2, step * 3, top] };
  }
}

// Tilt effect on cards
document.querySelectorAll(".card-wrapper").forEach(card => {
  card.addEventListener("mousemove", e => {
    if (window.matchMedia("(prefers-reduced-motion: reduce)").matches) return;
    if (card.classList.contains("flipped")) return;
    const rect = card.getBoundingClientRect();
    const x = e.clientX - rect.left - rect.width / 2;
    const y = e.clientY - rect.top - rect.height / 2;
    const tiltX = (y / (rect.height / 2)) * -2.5;
    const tiltY = (x / (rect.width / 2)) * 2.5;
    card.style.transform = `translateY(-4px) rotateX(${tiltX.toFixed(2)}deg) rotateY(${tiltY.toFixed(2)}deg)`;
  });
  card.addEventListener("mouseleave", () => {
    card.style.transform = "";
  });
});

// SAFE RENDER WRAPPER WITH ERROR BOUNDARY
function safeRender(name, cardId, fn) {
  try {
    fn();
  } catch (err) {
    console.error(`[SwiftBite] Error rendering ${name}:`, err);
    if (cardId) {
      const card = $(cardId);
      if (card) {
        const front = card.querySelector(".card-front");
        if (front) {
          const hl = front.querySelector(".card-headline");
          if (hl) hl.textContent = "Gagal memuat grafik";
        }
      }
    }
  }
}

// ================= MAIN RENDER FUNCTION =================
function render(isFilterTriggered) {
  syncDropdowns();
  syncTrendTabs();

  const f = filtered();
  const fM = state.month, fZ = state.zone, fS = state.sentiment;

  // Live Counter Animasi
  const cntEl = $("cntFiltered");
  const prevCount = cntEl ? (parseInt(cntEl.textContent) || 0) : 0;
  const targetCount = f.length;
  setTxt("cntTotal", TOTAL_ALL_ORDERS);
  if (cntEl) {
    if (isFilterTriggered && prevCount !== targetCount) {
      const start = (typeof performance !== 'undefined' && performance.now) ? performance.now() : Date.now();
      const duration = 250;
      function animateCounter(now) {
        const curTime = (typeof performance !== 'undefined' && performance.now) ? performance.now() : (now || Date.now());
        const elapsed = curTime - start;
        const progress = Math.min(elapsed / duration, 1);
        const cur = Math.round(prevCount + (targetCount - prevCount) * progress);
        cntEl.textContent = cur;
        if (progress < 1 && elapsed < duration) requestAnimationFrame(animateCounter);
        else cntEl.textContent = targetCount;
      }
      requestAnimationFrame(animateCounter);
    } else {
      cntEl.textContent = targetCount;
    }
  }

  // Blink/Highlight cards on filter change
  if (isFilterTriggered) {
    document.querySelectorAll(".card-wrapper").forEach(card => {
      card.classList.remove("card-highlight");
      void card.offsetWidth;
      card.classList.add("card-highlight");
      setTimeout(() => card.classList.remove("card-highlight"), 650);
    });
  }

  // Global KPIs calculation
  const orders = f.length;
  const net = f.reduce((s, d) => s + (d.np || 0), 0);
  const rev = f.filter(d => d.hr === 1).length;
  const dv = f.filter(d => d.dv === 1).length;
  const late = f.filter(d => d.dv === 1 && d.lt === 1).length;
  const rated = f.filter(d => d.dv === 1 && d.rt !== null && d.rt !== undefined && !isNaN(d.rt));
  const avg = rated.length ? (rated.reduce((s, d) => s + d.rt, 0) / rated.length) : null;

  // Monthly breakdown for Sparklines & Trend
  const mm = {
    "2026-01": { n: 0, np: 0, hr: 0, dv: 0, late: 0, rt: [], fail: 0, ref: 0 },
    "2026-02": { n: 0, np: 0, hr: 0, dv: 0, late: 0, rt: [], fail: 0, ref: 0 },
    "2026-03": { n: 0, np: 0, hr: 0, dv: 0, late: 0, rt: [], fail: 0, ref: 0 }
  };

  f.forEach(d => {
    if (mm[d.m]) {
      mm[d.m].n++;
      mm[d.m].np += (d.np || 0);
      if (d.hr === 1) mm[d.m].hr++;
      if (d.dv === 1) {
        mm[d.m].dv++;
        if (d.lt === 1) mm[d.m].late++;
        if (d.rt !== null && d.rt !== undefined && !isNaN(d.rt)) mm[d.m].rt.push(d.rt);
      }
      const cs = (d.cs || "").toUpperCase();
      if (cs === "FAILED") mm[d.m].fail++;
      if (cs === "REFUNDED") mm[d.m].ref++;
    }
  });

  // RENDER SECTIONS SAFELY (No single failure stops others)
  safeRender("KPI Cards", null, () => renderKPIs(orders, net, rev, dv, late, rated, avg, mm));
  safeRender("Trend Chart", "card-chart-trend", () => renderTrendChart(mm, orders, net, fM));
  safeRender("Donut Sentiment", "card-chart-sent", () => renderDonut(f, orders, rev));
  safeRender("BQ-04 Delay vs Rating", "card-bq04", () => renderBQ04(f));
  safeRender("Zones Late Rate", "card-zones", () => renderZones(f, dv, late));
  safeRender("Order Status Gauge", "card-status", () => renderGauge(f, orders, dv, mm, fM));
  safeRender("Ringkasan Temuan", "card-insights", () => renderInsights(f, orders, net, rev, dv, late, rated, avg));
  safeRender("Sidebar Summary", null, () => renderSidebarSummary(orders, dv, late, rev));
}

// 1. KPI CARDS
function renderKPIs(orders, net, rev, dv, late, rated, avg, mm) {
  setTxt("k1", orders);
  setTxt("k2", fmtMoney(net));
  setTxt("k3", orders ? (rnd(100 * rev / orders, 1).replace('.', ',') + "%") : "N/A");
  setTxt("k4", dv ? (rnd(100 * late / dv, 1).replace('.', ',') + "%") : "N/A");
  setTxt("k5", avg !== null ? rnd(avg, 2).replace('.', ',') : "N/A");

  // Sparklines
  const spOrders = ["2026-01", "2026-02", "2026-03"].map(m => mm[m].n);
  const spNet = ["2026-01", "2026-02", "2026-03"].map(m => mm[m].np);
  const spCov = ["2026-01", "2026-02", "2026-03"].map(m => mm[m].n ? (100 * mm[m].hr / mm[m].n) : null);
  const spLate = ["2026-01", "2026-02", "2026-03"].map(m => mm[m].dv ? (100 * mm[m].late / mm[m].dv) : null);
  const spRating = ["2026-01", "2026-02", "2026-03"].map(m => mm[m].rt.length ? (mm[m].rt.reduce((s,v)=>s+v,0) / mm[m].rt.length) : null);

  drawSparkline("sp1", spOrders, "#E8590C");
  drawSparkline("sp2", spNet, "#1B9E6B");
  drawSparkline("sp3", spCov, "#2563EB");
  drawSparkline("sp4", spLate, "#DC2626");
  drawSparkline("sp5", spRating, "#D97706");

  // KPI Subtitles
  setTxt("k3s", `${rev} / ${orders} order`);
  setTxt("k4s", `${late} / ${dv} valid`);
  setTxt("k5s", `n = ${rated.length} review`);

  // Back Flashcards KPI Formula texts
  setTxt("bk1-f", `${orders} pesanan`);
  setTxt("bk2-f", fmtMoney(net));
  setTxt("bk3-f", `${rev} dari ${orders} pesanan`);
  setTxt("bk4-f", `${late} dari ${dv} pesanan terkirim`);
  setTxt("bk5-f", `Total skor dari ${rated.length} ulasan`);
}

// 2. TREND CHART: DUA PANEL BERDAMPINGAN DENGAN SVG
function renderTrendChart(mmData, totOrders, totNet, activeMonth) {
  const ml = ["2026-01", "2026-02", "2026-03"];
  const mLabels = ["Jan 2026", "Feb 2026", "Mar 2026"];

  const pLeftChart = $("panelLeftChart");
  const pRightChart = $("panelRightChart");
  if (!pLeftChart || !pRightChart) return;

  const netVals = ml.map(m => mmData[m].np / 1e6);
  const ordVals = ml.map(m => mmData[m].n);
  const cancRates = ml.map(m => mmData[m].n ? (100 * (mmData[m].fail + mmData[m].ref) / mmData[m].n) : 0);
  const totalCanc = ml.reduce((s, m) => s + mmData[m].fail + mmData[m].ref, 0);
  const avgCancRate = totOrders > 0 ? (100 * totalCanc / totOrders) : 0;

  // Update Summary Strip (4 Kolom)
  let maxNet = -1, peakNetM = null;
  let maxOrd = -1, peakOrdM = null;
  let maxCancRate = -1, peakCancM = null;

  ml.forEach(m => {
    const o = mmData[m];
    if (o.np > maxNet) { maxNet = o.np; peakNetM = m; }
    if (o.n > maxOrd) { maxOrd = o.n; peakOrdM = m; }
    const canc = o.fail + o.ref;
    const crate = o.n ? (100 * canc / o.n) : -1;
    if (crate > maxCancRate) { maxCancRate = crate; peakCancM = m; }
  });

  const mName = k => k === "2026-01" ? "Jan" : k === "2026-02" ? "Feb" : "Mar";
  setTxt("stPeakNet", (maxNet > 0 && peakNetM) ? `${mName(peakNetM)}: Rp${(maxNet/1e6).toFixed(1).replace('.', ',')} jt` : "N/A");
  setTxt("stPeakOrd", (maxOrd > 0 && peakOrdM) ? `${mName(peakOrdM)}: ${maxOrd} pesanan` : "N/A");
  setTxt("stPeakCanc", (maxCancRate >= 0 && peakCancM) ? `${mName(peakCancM)}: ${rnd(maxCancRate, 1).replace('.', ',')}%` : "N/A");
  setTxt("stTotalPeriod", (totOrders > 0) ? `${fmtMoney(totNet)} dan ${totOrders} pesanan` : "N/A");

  if (activeTrendTab === "combo") {
    setTxt("panelLeftTitle", "Net Payment (Rp juta)");
    setTxt("panelLeftTotal", totOrders > 0 ? `Rp${(totNet/1e6).toFixed(2).replace('.',',')} jt` : "N/A");
    setTxt("panelLeftSub", "Total periode");

    setTxt("panelRightTitle", "Jumlah pesanan");
    setTxt("panelRightTotal", totOrders);
    setTxt("panelRightSub", "Total pesanan");

    const netMax = Math.max(...netVals, 0);
    const ordMax = Math.max(...ordVals, 0);

    const scaleNet = getNiceScale(netMax, true);
    const scaleOrd = getNiceScale(ordMax, false);

    pLeftChart.innerHTML = buildBarChartSVG({
      vals: netVals,
      ticks: scaleNet.ticks,
      topScale: scaleNet.top,
      labels: mLabels,
      valFormatter: v => `Rp${rnd(v, 1).replace('.', ',')} jt`,
      gradId: "gradNet",
      gradStops: [["0%", "#F59F42"], ["100%", "#E8590C"]],
      calcDiff: (cur, prev) => prev > 0 ? ((cur - prev) / prev) * 100 : null,
      isInverseChip: false
    });

    pRightChart.innerHTML = buildBarChartSVG({
      vals: ordVals,
      ticks: scaleOrd.ticks,
      topScale: scaleOrd.top,
      labels: mLabels,
      valFormatter: v => String(v),
      gradId: "gradOrd",
      gradStops: [["0%", "#8A5A3C"], ["100%", "#2B1A10"]],
      calcDiff: (cur, prev) => prev > 0 ? ((cur - prev) / prev) * 100 : null,
      isInverseChip: false
    });
  } else {
    setTxt("panelLeftTitle", "Tingkat Pembatalan (%)");
    setTxt("panelLeftTotal", totOrders > 0 ? `${rnd(avgCancRate, 1).replace('.', ',')}%` : "N/A");
    setTxt("panelLeftSub", "Rata-rata periode");

    setTxt("panelRightTitle", "Komposisi Pembatalan");
    setTxt("panelRightTotal", totalCanc);
    setTxt("panelRightSub", "Total pembatalan");

    const maxRate = Math.max(...cancRates, 0);
    const scaleRate = { top: Math.max(20, Math.ceil(maxRate / 5) * 5), ticks: [0, 5, 10, 15, 20] };

    pLeftChart.innerHTML = buildBarChartSVG({
      vals: cancRates,
      ticks: scaleRate.ticks,
      topScale: scaleRate.top,
      labels: mLabels,
      valFormatter: v => `${rnd(v, 1).replace('.', ',')}%`,
      gradId: "gradCanc",
      gradStops: [["0%", "#EF4444"], ["100%", "#DC2626"]],
      calcDiff: (cur, prev) => cur - prev,
      isInverseChip: true
    });

    pRightChart.innerHTML = buildBarChartSVG({
      vals: [mmData["2026-01"].fail + mmData["2026-01"].ref, mmData["2026-02"].fail + mmData["2026-02"].ref, mmData["2026-03"].fail + mmData["2026-03"].ref],
      ticks: [0, 5, 10, 15, 20],
      topScale: 20,
      labels: mLabels,
      valFormatter: v => String(v),
      gradId: "gradCancN",
      gradStops: [["0%", "#B91C1C"], ["100%", "#7F1D1D"]],
      calcDiff: (cur, prev) => cur - prev,
      isInverseChip: true
    });
  }

  // HEADLINE DEPAN & STRIP BELAKANG: TREN BULANAN
  const sFinding = $("stripTrendFinding");
  const sBusiness = $("stripTrendBusiness");
  const sHeadline = $("headlineTrend");

  if (totOrders === 0) {
    if (sHeadline) sHeadline.textContent = "Belum ada pesanan pada filter ini.";
    if (sFinding) sFinding.textContent = "Belum ada data pada filter ini";
    if (sBusiness) sBusiness.textContent = "Ubah kombinasi filter untuk menelaah tren bulanan.";
  } else if (activeMonth !== "ALL") {
    const curNet = (mmData[activeMonth].np / 1e6).toFixed(1).replace('.', ',');
    const curOrd = mmData[activeMonth].n;
    if (sHeadline) sHeadline.textContent = `Bulan terpilih: Rp${curNet} jt dari ${curOrd} pesanan tercatat.`;
    if (sFinding) sFinding.textContent = `Bulan ${activeMonth} membukukan Net Payment Rp${curNet} jt dari ${curOrd} pesanan.`;
    if (sBusiness) sBusiness.textContent = "Fokus operasional pada kapasitas dapur dan kecepatan kurir di bulan aktif.";
  } else {
    const diffNet = netVals[0] > 0 ? (((netVals[2] - netVals[0]) / netVals[0]) * 100).toFixed(1).replace('.', ',') : "0";
    if (sHeadline) sHeadline.textContent = `Net Payment naik +${diffNet}% di Maret (Rp${rnd(netVals[2], 1).replace('.', ',')} jt) dipicu pertumbuhan pesanan.`;
    if (sFinding) sFinding.textContent = `Net Payment tumbuh dari Rp${rnd(netVals[0], 1).replace('.', ',')} jt (Jan) ke Rp${rnd(netVals[2], 1).replace('.', ',')} jt (Mar), sejalan dengan kenaikan pesanan dari ${ordVals[0]} ke ${ordVals[2]}.`;
    if (sBusiness) sBusiness.textContent = "Pertumbuhan volume pesanan mendorong ekspansi omzet; mitigasi beban operasional kurir saat jam puncak.";
  }
}

// SVG Builder for Bar Chart (Headroom ~44px, H=200, baseLineY=162, topY=44)
function buildBarChartSVG(cfg) {
  const { vals, ticks, topScale, labels, valFormatter, gradId, gradStops, calcDiff, isInverseChip } = cfg;
  const W = 380, H = 200;
  const padL = 40, padR = 15, baseLineY = 162, topY = 44;
  const chartH = baseLineY - topY;

  let gridSvg = "";
  for (let i = 0; i < ticks.length; i++) {
    const y = baseLineY - (i / (ticks.length - 1)) * chartH;
    gridSvg += `<line x1="${padL}" y1="${y}" x2="${W - padR}" y2="${y}" stroke="#F3E5D8" stroke-width="1"/>`;
    gridSvg += `<text x="${padL - 6}" y="${y + 3.5}" font-size="10" font-weight="600" fill="#7C6F64" text-anchor="end">${ticks[i]}</text>`;
  }

  const barCenters = [padL + 45, padL + 145, padL + 245];
  const barW = 44;

  let barsSvg = "";
  vals.forEach((val, i) => {
    const cx = barCenters[i];
    const barH = topScale > 0 ? Math.min(chartH, Math.max(0, (val / topScale) * chartH)) : 0;
    const y = baseLineY - barH;

    let chipSvg = "";
    if (i > 0 && calcDiff) {
      const diff = calcDiff(val, vals[i - 1]);
      if (diff !== null && !isNaN(diff)) {
        const isUp = diff >= 0;
        const arrow = isUp ? "↑" : "↓";
        const sign = isUp ? "+" : "";
        const diffText = `${arrow} ${sign}${diff.toFixed(1).replace('.', ',')}%`;

        let chipBg = isUp ? "#E6F4EA" : "#FCE8E6";
        let chipColor = isUp ? "#137333" : "#C5221F";
        if (isInverseChip) {
          chipBg = isUp ? "#FCE8E6" : "#E6F4EA";
          chipColor = isUp ? "#C5221F" : "#137333";
        }
        const chipY = Math.max(16, y - 25);
        chipSvg = `
          <g transform="translate(${cx}, ${chipY})">
            <rect x="-32" y="-9" width="64" height="17" rx="8.5" fill="${chipBg}" />
            <text x="0" y="3.5" font-size="9" font-weight="700" fill="${chipColor}" text-anchor="middle">${diffText}</text>
          </g>
        `;
      }
    }

    // Label nilai bar: selalu di atas bar, font-size 13px, tebal 700, teks gelap #2B1A10
    const valY = y - 7;
    const valColor = "#2B1A10";

    barsSvg += `
      ${chipSvg}
      <rect x="${cx - barW / 2}" y="${y}" width="${barW}" height="${barH}" rx="6" fill="url(#${gradId})" />
      <text x="${cx}" y="${valY}" font-size="13" font-weight="700" fill="${valColor}" text-anchor="middle">${valFormatter(val)}</text>
      <text x="${cx}" y="${baseLineY + 18}" font-size="11.5" font-weight="600" fill="#7C6F64" text-anchor="middle">${labels[i]}</text>
    `;
  });

  return `
    <svg viewBox="0 0 ${W} ${H}" width="100%" height="100%" preserveAspectRatio="xMidYMid meet">
      <defs>
        <linearGradient id="${gradId}" x1="0" y1="0" x2="0" y2="1">
          <stop offset="${gradStops[0][0]}" stop-color="${gradStops[0][1]}" />
          <stop offset="${gradStops[1][0]}" stop-color="${gradStops[1][1]}" />
        </linearGradient>
      </defs>
      ${gridSvg}
      ${barsSvg}
    </svg>
  `;
}

// 3. DONUT SENTIMEN REVIEW (BQ-03)
function renderDonut(f, orders, rev) {
  const counts = { "Positive": 0, "Neutral": 0, "Negative": 0, "No Text": 0, "Unknown": 0 };
  let totalRev = 0;
  f.forEach(d => {
    if (d.hr === 1) {
      totalRev++;
      if (counts[d.s] !== undefined) counts[d.s]++;
      else counts["Unknown"]++;
    }
  });

  setTxt("donutTotalText", totalRev);

  const colors = {
    "Positive": "#1B9E6B",
    "Neutral": "#F59F42",
    "Negative": "#DC2626",
    "No Text": "#6B7280",
    "Unknown": "#9CA3AF"
  };

  const svg = $("svgDonut");
  if (svg) {
    svg.innerHTML = "";
    const cx = 80, cy = 80, r = 58;
    const circ = 2 * Math.PI * r;

    if (totalRev === 0) {
      const bgCirc = document.createElementNS("http://www.w3.org/2000/svg", "circle");
      bgCirc.setAttribute("cx", cx);
      bgCirc.setAttribute("cy", cy);
      bgCirc.setAttribute("r", r);
      bgCirc.setAttribute("fill", "none");
      bgCirc.setAttribute("stroke", "#F3E5D8");
      bgCirc.setAttribute("stroke-width", "22");
      svg.appendChild(bgCirc);
    } else {
      let offset = 0;
      SENT_ORDER.forEach(k => {
        const cnt = counts[k];
        if (cnt > 0) {
          const dash = (cnt / totalRev) * circ;
          const circle = document.createElementNS("http://www.w3.org/2000/svg", "circle");
          circle.setAttribute("cx", cx);
          circle.setAttribute("cy", cy);
          circle.setAttribute("r", r);
          circle.setAttribute("fill", "none");
          circle.setAttribute("stroke", colors[k]);
          circle.setAttribute("stroke-width", "22");
          circle.setAttribute("stroke-dasharray", `${dash} ${circ - dash}`);
          circle.setAttribute("stroke-dashoffset", -offset);
          circle.setAttribute("transform", `rotate(-90 ${cx} ${cy})`);
          circle.innerHTML = `<title>${k}: ${cnt} (${(cnt/totalRev*100).toFixed(1)}%)</title>`;
          svg.appendChild(circle);
          offset += dash;
        }
      });
    }
  }

  // Donut Legend: 1 baris per label (titik warna, nama, jumlah, dan persen dalam satu baris)
  const leg = $("donutLegend");
  if (leg) {
    leg.innerHTML = "";
    SENT_ORDER.forEach(k => {
      const cnt = counts[k];
      const pct = totalRev > 0 ? (100 * cnt / totalRev) : 0;
      const isUnknown = k === "Unknown";
      const dotColor = isUnknown ? "#9CA3AF" : colors[k];
      const nameColor = isUnknown ? "#9CA3AF" : "#2B1A10";
      const valColor = isUnknown ? "#9CA3AF" : "#2B1A10";

      const row = document.createElement("div");
      row.className = "donut-legend-item";
      row.innerHTML = `
        <div class="legend-left">
          <span class="legend-dot" style="background:${dotColor};"></span>
          <span class="legend-name" style="color:${nameColor};">${k}</span>
        </div>
        <span class="legend-val" style="color:${valColor};">${cnt} (${rnd(pct, 1).replace('.', ',')}%)</span>
      `;
      leg.appendChild(row);
    });
  }

  // HEADLINE DEPAN & STRIP BELAKANG: SENTIMEN REVIEW
  const sFinding = $("stripSentFinding");
  const sBusiness = $("stripSentBusiness");
  const sHeadline = $("headlineSent");

  if (totalRev === 0) {
    if (sHeadline) sHeadline.textContent = "Belum ada ulasan pada filter ini.";
    if (sFinding) sFinding.textContent = "Belum ada data pada filter ini";
    if (sBusiness) sBusiness.textContent = "Ubah filter untuk melihat telaah sentimen ulasan.";
  } else {
    const pPos = rnd(100 * counts["Positive"] / totalRev, 1).replace('.', ',');
    const covPct = orders > 0 ? rnd(100 * totalRev / orders, 1).replace('.', ',') : "0,0";
    if (sHeadline) sHeadline.textContent = `Positive memimpin ${pPos}%, review coverage baru mencapai ${covPct}%.`;
    if (sFinding) sFinding.textContent = `Sentimen Positive mendominasi (${pPos}%), diikuti Neutral (${rnd(100 * counts["Neutral"] / totalRev, 1).replace('.', ',')}%) dan Negative (${rnd(100 * counts["Negative"] / totalRev, 1).replace('.', ',')}%).`;
    if (sBusiness) sBusiness.textContent = "Mayoritas pembeli memberikan penilaian baik, namun perlu insentif untuk menaikkan review coverage.";
  }
}

// 4. KETERLAMBATAN & RATING (BQ-04)
function renderBQ04(f) {
  const rated = f.filter(d => d.dv === 1 && d.rt !== null && d.rt !== undefined && !isNaN(d.rt));
  const late = rated.filter(d => d.lt === 1);
  const ontime = rated.filter(d => d.lt === 0);

  const lateN = late.length;
  let lateAvg = null, lateMed = null, lateNegCount = 0;
  if (lateN > 0) {
    const scores = late.map(d => d.rt).sort((a, b) => a - b);
    const sum = scores.reduce((a, b) => a + b, 0);
    lateAvg = sum / lateN;
    const mid = Math.floor(lateN / 2);
    lateMed = (lateN % 2 !== 0) ? scores[mid] : ((scores[mid - 1] + scores[mid]) / 2);
    lateNegCount = late.filter(d => d.s === "Negative").length;

    setTxt("bq4LateAvg", rnd(lateAvg, 2).replace('.', ','));
    setTxt("bq4LateMed", rnd(lateMed, 1).replace('.', ','));
    setTxt("bq4LateN", `n = ${lateN} ulasan`);
    const pLateNeg = rnd(100 * lateNegCount / lateN, 1).replace('.', ',');
    setTxt("bq4LateNeg", `${pLateNeg}% (${lateNegCount}/${lateN})`);
    const barLate = $("bq4BarLate");
    if (barLate) barLate.style.width = `${Math.min(100, Math.max(0, 100 * lateNegCount / lateN))}%`;

    const rBarLate = $("bq4RatingBarLate");
    if (rBarLate) rBarLate.style.width = `${Math.min(100, Math.max(0, (lateAvg / 5) * 100))}%`;
    setTxt("bq4BarLateVal", rnd(lateAvg, 2).replace('.', ','));
  } else {
    setTxt("bq4LateAvg", "N/A");
    setTxt("bq4LateMed", "N/A");
    setTxt("bq4LateN", "n = 0 ulasan");
    setTxt("bq4LateNeg", "N/A (0/0)");
    const barLate = $("bq4BarLate");
    if (barLate) barLate.style.width = "0%";
    const rBarLate = $("bq4RatingBarLate");
    if (rBarLate) rBarLate.style.width = "0%";
    setTxt("bq4BarLateVal", "N/A");
  }

  const ontimeN = ontime.length;
  let ontimeAvg = null, ontimeMed = null, ontimeNegCount = 0;
  if (ontimeN > 0) {
    const scores = ontime.map(d => d.rt).sort((a, b) => a - b);
    const sum = scores.reduce((a, b) => a + b, 0);
    ontimeAvg = sum / ontimeN;
    const mid = Math.floor(ontimeN / 2);
    ontimeMed = (ontimeN % 2 !== 0) ? scores[mid] : ((scores[mid - 1] + scores[mid]) / 2);
    ontimeNegCount = ontime.filter(d => d.s === "Negative").length;

    setTxt("bq4OntimeAvg", rnd(ontimeAvg, 2).replace('.', ','));
    setTxt("bq4OntimeMed", rnd(ontimeMed, 1).replace('.', ','));
    setTxt("bq4OntimeN", `n = ${ontimeN} ulasan`);
    const pOntimeNeg = rnd(100 * ontimeNegCount / ontimeN, 1).replace('.', ',');
    setTxt("bq4OntimeNeg", `${pOntimeNeg}% (${ontimeNegCount}/${ontimeN})`);
    const barOntime = $("bq4BarOntime");
    if (barOntime) barOntime.style.width = `${Math.min(100, Math.max(0, 100 * ontimeNegCount / ontimeN))}%`;

    const rBarOntime = $("bq4RatingBarOntime");
    if (rBarOntime) rBarOntime.style.width = `${Math.min(100, Math.max(0, (ontimeAvg / 5) * 100))}%`;
    setTxt("bq4BarOntimeVal", rnd(ontimeAvg, 2).replace('.', ','));
  } else {
    setTxt("bq4OntimeAvg", "N/A");
    setTxt("bq4OntimeMed", "N/A");
    setTxt("bq4OntimeN", "n = 0 ulasan");
    setTxt("bq4OntimeNeg", "N/A (0/0)");
    const barOntime = $("bq4BarOntime");
    if (barOntime) barOntime.style.width = "0%";
    const rBarOntime = $("bq4RatingBarOntime");
    if (rBarOntime) rBarOntime.style.width = "0%";
    setTxt("bq4BarOntimeVal", "N/A");
  }

  // HEADLINE DEPAN & STRIP BELAKANG: BQ-04
  const sFinding = $("stripBQ04Finding");
  const sBusiness = $("stripBQ04Business");
  const sHeadline = $("headlineBQ04");

  if (rated.length === 0) {
    if (sHeadline) sHeadline.textContent = "Belum ada pesanan dengan rating pada filter ini.";
    if (sFinding) sFinding.textContent = "Belum ada data pada filter ini";
    if (sBusiness) sBusiness.textContent = "Ubah filter untuk membandingkan kepuasan pesanan tepat waktu vs terlambat.";
    return;
  }

  if (lateAvg !== null && ontimeAvg !== null) {
    const lStr = rnd(lateAvg, 2).replace('.', ',');
    const oStr = rnd(ontimeAvg, 2).replace('.', ',');
    if (sHeadline) sHeadline.textContent = `Pesanan tepat waktu meraih rating ${oStr} vs terlambat ${lStr}.`;
    if (sFinding) sFinding.textContent = `Pesanan tepat waktu meraih rating rata-rata ${oStr} (median ${rnd(ontimeMed,1).replace('.',',')}), sedangkan pesanan terlambat berada pada ${lStr} (median ${rnd(lateMed,1).replace('.',',')}). Porsi ulasan negatif 4x lebih tinggi pada pesanan terlambat.`;
    if (sBusiness) sBusiness.textContent = "Keterlambatan berkorelasi dengan persepsi negatif pelanggan; pengawasan SLA pengantaran menjaga loyalitas pemesan.";
  } else if (ontimeAvg !== null) {
    if (sHeadline) sHeadline.textContent = `Seluruh pesanan berulasan terkirim tepat waktu (rating ${rnd(ontimeAvg, 2).replace('.', ',')}).`;
    if (sFinding) sFinding.textContent = `Seluruh ${ontimeN} pesanan terkirim tepat waktu dengan rata-rata rating ${rnd(ontimeAvg, 2).replace('.', ',')}.`;
    if (sBusiness) sBusiness.textContent = "Pertahankan konsistensi SLA ketepatan waktu pengantaran.";
  } else if (lateAvg !== null) {
    if (sHeadline) sHeadline.textContent = `Seluruh ulasan berasal dari pesanan terlambat (rating ${rnd(lateAvg, 2).replace('.', ',')}).`;
    if (sFinding) sFinding.textContent = `Seluruh ${lateN} ulasan tercatat terlambat dengan rata-rata rating ${rnd(lateAvg, 2).replace('.', ',')}.`;
    if (sBusiness) sBusiness.textContent = "Perlu investigasi operasional pengantaran untuk memulihkan kepuasan pemesan.";
  }
}

// 5. LATE RATE PER ZONA (BQ-02)
function renderZones(f, totalDv, totalLate) {
  const zStats = {};
  f.forEach(d => {
    if (!zStats[d.z]) zStats[d.z] = { dv: 0, late: 0 };
    if (d.dv === 1) {
      zStats[d.z].dv++;
      if (d.lt === 1) zStats[d.z].late++;
    }
  });

  const arr = Object.keys(zStats).map(z => ({
    name: z,
    dv: zStats[z].dv,
    late: zStats[z].late,
    rate: zStats[z].dv ? (100 * zStats[z].late / zStats[z].dv) : 0
  })).sort((a, b) => b.rate - a.rate);

  const container = $("zoneRankList");
  if (container) {
    container.innerHTML = "";
    if (arr.length === 0 || arr.every(z => z.dv === 0)) {
      container.innerHTML = '<div style="color:var(--text-muted); font-size:12px; padding:12px 0;">Belum ada pesanan terkirim valid pada filter ini.</div>';
    } else {
      arr.forEach(z => {
        const item = document.createElement("div");
        item.className = "zone-rank-item";
        item.innerHTML = `
          <div class="zone-rank-head">
            <span class="zone-name">${z.name}</span>
            <span class="zone-val">${rnd(z.rate, 1).replace('.', ',')}% (${z.late}/${z.dv})</span>
          </div>
          <div class="zone-bar-track">
            <div class="zone-bar-fill" style="width: ${Math.min(100, Math.max(0, z.rate))}%;"></div>
          </div>
        `;
        container.appendChild(item);
      });
    }
  }

  // HEADLINE DEPAN & STRIP BELAKANG: ZONA
  const sFinding = $("stripZoneFinding");
  const sBusiness = $("stripZoneBusiness");
  const sHeadline = $("headlineZone");

  if (arr.length === 0 || arr.every(z => z.dv === 0)) {
    if (sHeadline) sHeadline.textContent = "Belum ada pesanan terkirim pada filter ini.";
    if (sFinding) sFinding.textContent = "Belum ada data pada filter ini";
    if (sBusiness) sBusiness.textContent = "Ubah filter untuk melihat performa pengiriman antar zona.";
  } else {
    const worst = arr[0];
    if (sHeadline) sHeadline.textContent = `${worst.name} tertinggi: ${rnd(worst.rate, 1).replace('.', ',')}% pesanan terkirim terlambat.`;
    if (sFinding) sFinding.textContent = `Zona ${worst.name} memiliki keterlambatan tertinggi (${rnd(worst.rate, 1).replace('.', ',')}%, ${worst.late}/${worst.dv} pesanan), disusul zona lainnya.`;
    if (sBusiness) sBusiness.textContent = "Evaluasi ketersediaan armada kurir dan rute pengantaran di zona dengan tingkat keterlambatan tinggi.";
  }
}

// 6. ORDER STATUS & GAUGE (BQ-05)
function renderGauge(f, orders, dv, mm, activeMonth) {
  const complete = dv;
  const failed = f.filter(d => (d.cs || "").toUpperCase() === "FAILED").length;
  const refunded = f.filter(d => (d.cs || "").toUpperCase() === "REFUNDED").length;
  const cancTotal = failed + refunded;

  const pctComp = orders ? (100 * complete / orders) : 0;
  const pctFail = orders ? (100 * failed / orders) : 0;
  const pctRef = orders ? (100 * refunded / orders) : 0;

  setTxt("gaugePctComp", rnd(pctComp, 1).replace('.', ',') + "%");
  setTxt("gaugeCntComp", `${complete} selesai`);
  setTxt("gaugePctFail", rnd(pctFail, 1).replace('.', ',') + "%");
  setTxt("gaugeCntFail", `${failed} failed`);
  setTxt("gaugePctRef", rnd(pctRef, 1).replace('.', ',') + "%");
  setTxt("gaugeCntRef", `${refunded} refunded`);

  // Semicircle Gauge SVG
  const svg = $("svgGauge");
  if (svg) {
    svg.innerHTML = "";
    const cx = 90, cy = 82, r = 62, w = 18;
    const arcLen = Math.PI * r;

    // Background track arc
    const bgArc = document.createElementNS("http://www.w3.org/2000/svg", "path");
    bgArc.setAttribute("d", `M ${cx - r} ${cy} A ${r} ${r} 0 0 1 ${cx + r} ${cy}`);
    bgArc.setAttribute("fill", "none");
    bgArc.setAttribute("stroke", "#F3E5D8");
    bgArc.setAttribute("stroke-width", w);
    svg.appendChild(bgArc);

    if (orders > 0) {
      const segs = [
        { pct: pctComp, color: "#1B9E6B" },
        { pct: pctFail, color: "#DC2626" },
        { pct: pctRef, color: "#F59F42" }
      ];
      let offset = 0;
      segs.forEach(s => {
        if (s.pct > 0) {
          const sLen = (s.pct / 100) * arcLen;
          const arc = document.createElementNS("http://www.w3.org/2000/svg", "path");
          arc.setAttribute("d", `M ${cx - r} ${cy} A ${r} ${r} 0 0 1 ${cx + r} ${cy}`);
          arc.setAttribute("fill", "none");
          arc.setAttribute("stroke", s.color);
          arc.setAttribute("stroke-width", w);
          arc.setAttribute("stroke-dasharray", `${sLen} ${arcLen - sLen}`);
          arc.setAttribute("stroke-dashoffset", -offset);
          svg.appendChild(arc);
          offset += sLen;
        }
      });
    }
  }

  // HEADLINE DEPAN & STRIP BELAKANG: STATUS PESANAN
  const sFinding = $("stripStatusFinding");
  const sBusiness = $("stripStatusBusiness");
  const sHeadline = $("headlineStatus");

  if (orders === 0) {
    if (sHeadline) sHeadline.textContent = "Belum ada pesanan pada filter ini.";
    if (sFinding) sFinding.textContent = "Belum ada data pada filter ini";
    if (sBusiness) sBusiness.textContent = "Ubah filter untuk melihat rasio penyelesaian dan pembatalan.";
  } else {
    if (sHeadline) sHeadline.textContent = `${rnd(pctComp, 1).replace('.', ',')}% pesanan selesai; pembatalan failed mendominasi di Maret.`;
    if (sFinding) sFinding.textContent = `Dari total ${orders} pesanan, ${complete} (${rnd(pctComp, 1).replace('.', ',')}%) selesai terkirim valid. Pembatalan tercatat ${cancTotal} pesanan (${failed} failed, ${refunded} refunded).`;
    if (sBusiness) sBusiness.textContent = "Tingkat pembatalan perlu dimonitor terutama kegagalan sistem pembayaran (failed) agar transaksi tidak terbuang sia-sia.";
  }
}

// 7. RINGKASAN TEMUAN (F-01 s/d F-05)
function renderInsights(f, orders, net, rev, dv, late, rated, avg) {
  const container = $("insightsList");
  if (!container) return;
  container.innerHTML = "";

  if (orders === 0) {
    container.innerHTML = `
      <div class="finding-item" onclick="navigateMenu('transaksi')">
        <span class="finding-badge">F-01</span>
        <span class="finding-text">Belum ada data pada filter ini</span>
        <span class="finding-arrow">→</span>
      </div>
      <div class="finding-item" onclick="navigateMenu('pengiriman')">
        <span class="finding-badge">F-02</span>
        <span class="finding-text">Belum ada data pada filter ini</span>
        <span class="finding-arrow">→</span>
      </div>
      <div class="finding-item" onclick="navigateMenu('review')">
        <span class="finding-badge">F-03</span>
        <span class="finding-text">Belum ada data pada filter ini</span>
        <span class="finding-arrow">→</span>
      </div>
      <div class="finding-item" onclick="navigateMenu('review')">
        <span class="finding-badge">F-04</span>
        <span class="finding-text">Belum ada data pada filter ini</span>
        <span class="finding-arrow">→</span>
      </div>
      <div class="finding-item" onclick="navigateMenu('pembatalan')">
        <span class="finding-badge">F-05</span>
        <span class="finding-text">Belum ada data pada filter ini</span>
        <span class="finding-arrow">→</span>
      </div>
    `;
    return;
  }

  // F-01: Net Payment
  const mmNet = { "2026-01": 0, "2026-02": 0, "2026-03": 0 };
  f.forEach(d => {
    if (mmNet[d.m] !== undefined) mmNet[d.m] += (d.np || 0);
  });
  let f01 = "";
  if (state.month !== "ALL") {
    f01 = `Net Payment tercatat Rp${(net/1e6).toFixed(1).replace('.', ',')} jt dari ${orders} pesanan.`;
  } else {
    const janNet = (mmNet["2026-01"]/1e6).toFixed(1).replace('.', ',');
    const marNet = (mmNet["2026-03"]/1e6).toFixed(1).replace('.', ',');
    const dir = mmNet["2026-03"] >= mmNet["2026-01"] ? "naik" : "bergerak";
    f01 = `Net Payment ${dir} dari Rp${janNet} jt (Jan) ke Rp${marNet} jt (Mar).`;
  }

  // F-02: Pengiriman per Zona
  const zones = {};
  f.forEach(d => {
    if (!zones[d.z]) zones[d.z] = { dv: 0, late: 0 };
    if (d.dv === 1) {
      zones[d.z].dv++;
      if (d.lt === 1) zones[d.z].late++;
    }
  });
  const zArr = Object.keys(zones).map(z => ({
    name: z,
    dv: zones[z].dv,
    late: zones[z].late,
    rate: zones[z].dv ? (100 * zones[z].late / zones[z].dv) : 0
  })).filter(z => z.dv > 0);

  let f02 = "";
  if (zArr.length === 0) {
    f02 = "Belum ada data pada filter ini";
  } else {
    const worstZ = zArr.sort((a, b) => b.rate - a.rate)[0];
    f02 = `Zona ${worstZ.name} paling sering terlambat: ${rnd(worstZ.rate, 1).replace('.', ',')}% (${worstZ.late}/${worstZ.dv}).`;
  }

  // F-03: Review Coverage & Sentimen
  let f03 = "";
  const covPct = rnd(100 * rev / orders, 1).replace('.', ',');
  if (rev === 0) {
    f03 = `Review Coverage 0,0%; belum ada ulasan pada filter ini.`;
  } else {
    const counts = { "Positive": 0, "Neutral": 0, "Negative": 0, "No Text": 0 };
    f.forEach(d => {
      if (d.hr === 1 && counts[d.s] !== undefined) counts[d.s]++;
    });
    const topSent = ["Positive", "Neutral", "Negative"].map(k => ({
      name: k,
      cnt: counts[k],
      pct: 100 * counts[k] / rev
    })).sort((a, b) => b.cnt - a.cnt)[0];
    f03 = `Review Coverage ${covPct}%; sentimen ${topSent.name} dominan (${rnd(topSent.pct, 1).replace('.', ',')}%).`;
  }

  // F-04: Keterlambatan & Rating
  let f04 = "";
  const lateRated = f.filter(d => d.dv === 1 && d.lt === 1 && d.rt !== null && !isNaN(d.rt));
  const ontimeRated = f.filter(d => d.dv === 1 && d.lt === 0 && d.rt !== null && !isNaN(d.rt));
  if (lateRated.length > 0 && ontimeRated.length > 0) {
    const lAvg = lateRated.reduce((s, d) => s + d.rt, 0) / lateRated.length;
    const oAvg = ontimeRated.reduce((s, d) => s + d.rt, 0) / ontimeRated.length;
    const comp = lAvg < oAvg ? "lebih rendah" : "berbeda";
    f04 = `Rating pesanan terlambat ${comp} (${rnd(lAvg, 2).replace('.', ',')} vs ${rnd(oAvg, 2).replace('.', ',')}); ini korelasi.`;
  } else if (rated.length > 0) {
    f04 = `Rata-rata rating ulasan: ${rnd(avg, 2).replace('.', ',')} (n=${rated.length}); ini korelasi.`;
  } else {
    f04 = "Belum ada data pada filter ini";
  }

  // F-05: Pembatalan
  let f05 = "";
  const totalCanc = f.filter(d => d.cs === "FAILED" || d.cs === "REFUNDED").length;
  const failCount = f.filter(d => d.cs === "FAILED").length;
  const refCount = f.filter(d => d.cs === "REFUNDED").length;
  if (totalCanc === 0) {
    f05 = "Tidak ada pesanan batal pada filter ini.";
  } else if (state.month !== "ALL") {
    f05 = `Dari ${totalCanc} pesanan batal, ${failCount} failed dan ${refCount} refunded.`;
  } else {
    f05 = `Dari ${totalCanc} pesanan batal, ${failCount} failed dan ${refCount} refunded; failed naik di Maret.`;
  }

  container.innerHTML = `
    <div class="finding-item" onclick="navigateMenu('transaksi')">
      <span class="finding-badge">F-01</span>
      <span class="finding-text">${f01}</span>
      <span class="finding-arrow">→</span>
    </div>
    <div class="finding-item" onclick="navigateMenu('pengiriman')">
      <span class="finding-badge">F-02</span>
      <span class="finding-text">${f02}</span>
      <span class="finding-arrow">→</span>
    </div>
    <div class="finding-item" onclick="navigateMenu('review')">
      <span class="finding-badge">F-03</span>
      <span class="finding-text">${f03}</span>
      <span class="finding-arrow">→</span>
    </div>
    <div class="finding-item" onclick="navigateMenu('review')">
      <span class="finding-badge">F-04</span>
      <span class="finding-text">${f04}</span>
      <span class="finding-arrow">→</span>
    </div>
    <div class="finding-item" onclick="navigateMenu('pembatalan')">
      <span class="finding-badge">F-05</span>
      <span class="finding-text">${f05}</span>
      <span class="finding-arrow">→</span>
    </div>
  `;
}

// 8. SIDEBAR SUMMARY
function renderSidebarSummary(orders, dv, late, rev) {
  setTxt("sbOrders", orders);
  setTxt("sbLate", dv ? (rnd(100 * late / dv, 1).replace('.', ',') + "%") : "N/A");
  setTxt("sbCov", orders ? (rnd(100 * rev / orders, 1).replace('.', ',') + "%") : "N/A");
}

window.addEventListener("resize", () => {
  render(false);
});

// Initial Render
render(false);
</script>
</body>
</html>
"""

payload = json.dumps(records, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
html = (HTML_TEMPLATE.replace("__DATA__", payload)
                     .replace("__VARIANT__", VARIANT_ID))
dash_path = OUT_DASH / "swiftbite_capstone_dashboard.html"
dash_path.write_text(html, encoding="utf-8")
print("Dashboard:", dash_path, f"({dash_path.stat().st_size/1024:.0f} KB, {len(records)} record)")


In [ ]:
# --- Uji logika dashboard: hitung ulang KPI dari `records` (logika yang sama dengan JS) dan bandingkan dengan pandas ---
rec = pd.DataFrame(records)

def kpi_from_records(r):
    n = len(r); dv = int(r["dv"].sum()); late = int((r["lt"] * r["dv"]).sum())
    return {"orders": n, "net": round(float(r["np"].sum()), 2), "cov": None if n == 0 else 100 * int(r["hr"].sum()) / n,
            "late": None if dv == 0 else 100 * late / dv}

def kpi_from_pandas(d):
    k = kpis(d)
    return {"orders": k["total_orders"], "net": round(k["net_payment"], 2), "cov": k["review_coverage"], "late": k["late_delivery_rate"]}

def same(a, b):
    return all((a[k] is None and b[k] is None) or (a[k] is not None and b[k] is not None and abs(a[k] - b[k]) < 1e-6) for k in a)

tests = [("tanpa filter", rec, df)]
m0 = df.loc[df["month"] != "Unknown", "month"].iloc[0]
r0 = df["restaurant"].iloc[0]
tests.append((f"bulan={m0}", rec[rec["m"] == m0], df[df["month"] == m0]))
tests.append((f"restoran={r0}", rec[rec["r"] == r0], df[df["restaurant"] == r0]))
tests.append((f"bulan={m0} & restoran={r0}", rec[(rec["m"] == m0) & (rec["r"] == r0)], df[(df["month"] == m0) & (df["restaurant"] == r0)]))
tests.append(("sentimen=Negative", rec[rec["s"] == "Negative"], df[(df["has_review"] == 1) & (df["sentiment_label"] == "Negative")]))
tests.append(("kombinasi nol baris", rec[(rec["m"] == "1900-01")], df[df["month"] == "1900-01"]))
tests.append(("kombinasi tanpa data (N/A)", rec[(rec["m"] == "none") & (rec["z"] == "none")], df[(df["month"] == "none") & (df["customer_zone"] == "none")]))

rows = []
for name, a, b in tests:
    ka = kpi_from_records(a); kb = kpi_from_pandas(b) if len(b) else {"orders": 0, "net": 0.0, "cov": None, "late": None}
    rows.append({"test_filter": name, "rows": len(a), "orders": ka["orders"], "late_rate": ka["late"], "match_pandas": same(ka, kb)})
dash_test = pd.DataFrame(rows)
display(dash_test)
EVIDENCE["dashboard_filter_tests"] = dash_test.to_dict("records")
assert dash_test["match_pandas"].all(), "Ada KPI dashboard yang tidak cocok dengan pandas."
assert pd.isna(dash_test.loc[dash_test["test_filter"] == "kombinasi tanpa data (N/A)", "late_rate"].iloc[0]), "Kombinasi tanpa data harus menghasilkan late_rate N/A (None)"
print("Catatan: kondisi nol baris harus tampil 'N/A' (rasio) di browser. Buka file HTML dan uji manual setiap filter.")


## Tahap 7 - Final QA

Tujuan: Memastikan seluruh komponen memenuhi standar kualitas (Quality Gate) sebelum dikumpulkan.